In [108]:
import pandas as pd
import numpy as np
import datetime as dt
import os
# import dask.dataframe as dd
import polars as pl

import requests
from bs4 import BeautifulSoup
import pyarrow.parquet as pq
import re
import pyarrow as pa
from IPython.display import display
from io import BytesIO
from pathlib import Path

import duckdb
from glob import glob

import warnings

# Отключаем проверку сертификатов SSL
requests.packages.urllib3.disable_warnings()

In [109]:
work_dir = '/Users/eduard/Documents/Code_projects/ML4RV/ML'

In [110]:
def normalize_columns(columns):
        return {re.sub(r"Cost of Capital.*", "Cost of Capital", col) for col in columns}

def find_start_row_in_sheet(sheet):
    """
    Функция находит индекс строки, содержащей все важные колонки.
    Возвращает номер строки, с которого начнется обработка.
    """
    required_columns = {"Industry Name", "Beta", "Cost of Equity", "Cost of Debt", "D/(D+E)", "Cost of Capital"}
    normalized_required_columns = normalize_columns(required_columns)
    for idx, row in sheet.iterrows():
        text_cells = [val.strip() for val in row.values if isinstance(val, str)]
        normalized_text_cells = normalize_columns(text_cells)
        # Проверяем наличие всех важных колонок в тексте строки
        if normalized_required_columns <= set(normalized_text_cells):
            return idx
    return None


def extract_data_from_wacc_file(url):
    """
    Функция принимает URL файла, загружает его и возвращает DataFrame с необходимыми преобразованиями,
    начиная с строки, содержащей 'Beta'.
    """
    response = requests.get(url, verify=False)
    if response.status_code != 200:
        raise ValueError(f"Ошибка при загрузке файла {url}, код статуса: {response.status_code}")

    # Чтение всех листов файла Excel
    try:
        sheets_dict = pd.read_excel(BytesIO(response.content), engine='xlrd', sheet_name=None)
    except Exception as e:
        raise ValueError(f"Ошибка при чтении Excel-файла {url}: {e}")

    # Пробегаемся по всем листам, пока не найдём подходящий
    found = False
    processed_df = None
    for sheet_name, sheet_data in sheets_dict.items():
        start_row_idx = find_start_row_in_sheet(sheet_data)
        if start_row_idx is not None:
            # Лист с валидной таблицей найден, работаем с ним
            processed_df = pd.read_excel(
                BytesIO(response.content),
                engine='xlrd',
                sheet_name=sheet_name,
                skiprows=start_row_idx + 1,
                header=0
            )
            found = True
            break

    if not found:
        raise ValueError(f"Подходящий лист с набором колонок не найден ни в одном листе файла {url}")

    # Извлечение country и year из имени файла
    filename = url.split('/')[-1]

    # Паттерн 1: wacc<страна><двух- или четырёхзначный год>.xls
    # Примеры: waccChina23.xls, waccChina2023.xls
    match_with_year = re.match(
        r'wacc(?P<country>[A-Za-z]+)(?P<year>\d{2,4})\.xls',
        filename,
        re.IGNORECASE
    )

    # Паттерн 2: wacc<страна>.xls → без года (например, waccChina.xls)
    match_no_year = re.match(r'wacc(?P<country>[A-Za-z]+)\.xls', filename, re.IGNORECASE)

    # Паттерн 3: wacc<двух- или четырёхзначный год>.xls → wacc23.xls, wacc2023.xls
    match_global_year = re.match(r'wacc(?P<year>\d{2,4})\.xls', filename, re.IGNORECASE)
    match_us = re.match(r'wacc.xls', filename, re.IGNORECASE)

    # Паттерн 4: wacc<страна>_<год>, wacc<страна>-<год> и т.п.
    match_with_sep = re.match(
        r'wacc(?P<country>[A-Za-z]+)[_\-\s](?P<year>\d{2,4})\.xls',
        filename,
        re.IGNORECASE
    )

    if match_with_year:
        country = match_with_year.group('country').lower()
        year_value = int(match_with_year.group('year'))
        # Если 2 цифры: 00–99 → интерпретируем как 19xx или 20xx
        if year_value < 100:
            full_year = year_value if year_value > 50 else 2000 + year_value
        else:
            full_year = year_value  # напрямую, если 2000 и выше

    elif match_no_year:
        country = match_no_year.group('country').lower()
        full_year = 2024  # по умолчанию, если год не указан

    elif match_us:
        country = 'us'
        full_year = 2024  # по умолчанию, если год не указан

    elif match_with_sep:
        country = match_with_sep.group('country').lower()
        year_value = int(match_with_sep.group('year'))
        if year_value < 100:
            full_year = year_value if year_value > 50 else 2000 + year_value
        else:
            full_year = year_value

    elif match_global_year:
        # Формат wacc23.xls или wacc2023.xls → считаем, что это us
        country = 'us'
        year_value = int(match_global_year.group('year'))
        if year_value < 100:
            full_year = year_value if year_value > 50 else 2000 + year_value
        else:
            full_year = year_value

    else:
        raise ValueError(f"Не удалось распознать имя файла: {filename}")

    # Добавляем колонки 'year' и 'country'
    processed_df['year'] = full_year
    processed_df['country'] = country

    # Рассчитываем D/E
    if 'E/(D+E)' in processed_df.columns:
        processed_df['D/E'] = 1 / processed_df['E/(D+E)'] - 1
    else:
        raise ValueError(f"Отсутствует колонка 'E/(D+E)' в файле {url}")

    # Оставляем нужные колонки
    columns_to_keep = ['country', 'year', 'Industry Name', 'Beta', 'Cost of Equity', 'Cost of Debt', 'Tax Rate', 'D/E', 'Cost of Capital']
    missing_cols = [col for col in columns_to_keep if col not in processed_df.columns]
    if missing_cols:
        raise ValueError(f"Отсутствуют необходимые колонки: {missing_cols} в файле {url}")

    processed_df = processed_df[columns_to_keep]

    # Переименовываем
    column_mapping = {
        'Industry Name': 'industry',
        'Beta': 'beta',
        'Cost of Equity': 'coe',
        'Cost of Debt': 'cod',
        'Tax Rate': 'tax_rate',
        'D/E': 'd_e',
        'Cost of Capital': 'wacc'
    }
    processed_df.rename(columns=column_mapping, inplace=True)

    return processed_df


def save_frame_as_parquet(df, output_dir, filename):
    """Сохранение DataFrame в parquet."""
    os.makedirs(output_dir, exist_ok=True)
    pq.write_table(pa.Table.from_pandas(df), os.path.join(output_dir, filename))


def process_multiple_files(base_url, filter_archives=False):
    """
    Обрабатывает все файлы wacc*.xls по указанному URL.
    Если filter_archives=True, то из этого каталога берутся только файлы С ГОДОМ.
    """
    base_url = base_url.strip().rstrip('/')  # убираем слэш в конце

    response = requests.get(base_url, verify=False)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    links = soup.find_all('a', href=True)

    # Фильтруем только wacc*.xls
    xls_links = [link['href'].strip('/') for link in links if re.match(r'^wacc.*\.xls$', link['href'])]

    # Дополнительная фильтрация: только с годом (две или четыре цифры), если это archives
    if filter_archives:
        # Оставляем только файлы, в имени которых есть хотя бы две цифры подряд (год)
        xls_links = [f for f in xls_links if re.search(r'\d{2}', f)]
        print(f"🔍 В режиме архивов отфильтровано: обрабатываются только файлы с годом. Найдено: {len(xls_links)}")

    dfs = []
    for file_name in xls_links:
        full_url = f"{base_url}/{file_name}"
        try:
            df = extract_data_from_wacc_file(full_url)
            dfs.append(df)
            # print(f"✅ Успешно обработан: {file_name}")
        except Exception as e:
            print(f"❌ Пропущен файл {file_name}: {e}")

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def get_wacc_data(parquet_dir):
    """
    Загружает данные WACC.
    :param parquet_dir: Путь к папке, где будет сохранён/найден wacc_data.parquet
    :return: pd.DataFrame
    """
    parquet_dir = Path(parquet_dir)
    parquet_path = parquet_dir / "wacc_data.parquet"
    print(parquet_path)

    # 1. Проверяем, есть ли кэш
    if parquet_path.exists():
        print(f"✅ Найден паркет с данными: {parquet_path}")
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f"❌ Ошибка чтения parquet: {e}. Перезагружаю...")

    # 2. Нет кэша — парсим
    urls = [
        'https://pages.stern.nyu.edu/~adamodar/pc/archives ',
        'https://pages.stern.nyu.edu/~adamodar/pc/datasets '
    ]

    all_dfs = []

    print(f"\n🔍 Обработка АРХИВОВ (только с годом): {urls[0]}")
    try:
        df_arch = process_multiple_files(urls[0], filter_archives=True)
        if not df_arch.empty:
            all_dfs.append(df_arch)
    except Exception as e:
        print(f"❌ Ошибка архивов: {e}")

    print(f"\n🔍 Обработка DATASETS (все файлы): {urls[1]}")
    try:
        df_ds = process_multiple_files(urls[1], filter_archives=False)
        if not df_ds.empty:
            all_dfs.append(df_ds)
    except Exception as e:
        print(f"❌ Ошибка datasets: {e}")

    if not all_dfs:
        print("⚠️ Ни один файл не загружен.")
        return pd.DataFrame()

    final_df = pd.concat(all_dfs, ignore_index=True)

    # 3. Сохраняем в указанную директорию
    try:
        parquet_dir.mkdir(parents=True, exist_ok=True)
        pq.write_table(pa.Table.from_pandas(final_df), parquet_path)
        print(f"✅ Данные сохранены: {parquet_path}")
    except Exception as e:
        print(f"❌ Не удалось сохранить файл: {e}")

    return final_df

In [10]:
wacc_data = get_wacc_data(f'{work_dir}/data/wacc')

/Users/eduard/Documents/Code_projects/ML4RV/ML/data/wacc/wacc_data.parquet

🔍 Обработка АРХИВОВ (только с годом): https://pages.stern.nyu.edu/~adamodar/pc/archives 
🔍 В режиме архивов отфильтровано: обрабатываются только файлы с годом. Найдено: 117
❌ Пропущен файл waccpassthrough13.xls: Подходящий лист с набором колонок не найден ни в одном листе файла https://pages.stern.nyu.edu/~adamodar/pc/archives/waccpassthrough13.xls
❌ Пропущен файл waccpassthrough14.xls: Подходящий лист с набором колонок не найден ни в одном листе файла https://pages.stern.nyu.edu/~adamodar/pc/archives/waccpassthrough14.xls
❌ Пропущен файл waccpassthrough15.xls: Подходящий лист с набором колонок не найден ни в одном листе файла https://pages.stern.nyu.edu/~adamodar/pc/archives/waccpassthrough15.xls
❌ Пропущен файл waccpassthrough16.xls: Подходящий лист с набором колонок не найден ни в одном листе файла https://pages.stern.nyu.edu/~adamodar/pc/archives/waccpassthrough16.xls
❌ Пропущен файл waccpassthrough17.xls: 

In [11]:
wacc_data = wacc_data.rename(columns={'country':'region'})

In [12]:
wacc_data['year'] = wacc_data['year'] + 1

### Country data

In [14]:
import glob
import re

path = 'data/wacc/country_erp_premia/'
files = glob.glob(path+"ctryprem*.xls*")
data = []

for f in files:
    year_match = re.search(r'ctryprem(?:July)?(\d{2})', f)
    year = int(year_match.group(1)) + 2000 if year_match else None
    df = pd.read_excel(f, sheet_name='Regional breakdown')
    df['Year'] = year
    if 'Region' not in df.columns:
        df['Region'] = None
    data.append(df[['Country', 'Country Risk Premium', 'Region', 'Year']])

country_data = pd.concat(data, ignore_index=True)
country_data.columns = ['Country', 'Country Risk Premium', 'Region', 'Year']

In [15]:
country_data = country_data.dropna(subset='Country')

In [16]:
country_data.to_parquet('data/wacc/country_data.parquet')

#### Переименовываем страны и унифицируем

In [17]:
country_data = pd.read_parquet('data/wacc/country_data.parquet')

In [18]:
countries_unified_mapping = {
'Abu Dhabi': 'Abu Dhabi',
'Albania': 'Albania',
'Andorra (Principality of)': 'Andorra (Principality of)',
'Angola': 'Angola',
'Argentina': 'Argentina',
'Armenia': 'Armenia',
'Aruba': 'Aruba',
'Australia': 'Australia',
'Austria': 'Austria',
'Azerbaijan': 'Azerbaijan',
'Bahamas': 'Bahamas',
'Bahrain': 'Bahrain',
'Bangladesh': 'Bangladesh',
'Barbados': 'Barbados',
'Belarus': 'Belarus',
'Belgium': 'Belgium',
'Belize': 'Belize',
'Bermuda': 'Bermuda',
'Bolivia': 'Bolivia',
'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
'Botswana': 'Botswana',
'Brazil': 'Brazil',
'Bulgaria': 'Bulgaria',
'Burkina Faso': 'Burkina',
'Cambodia': 'Cambodia',
'Cameroon': 'Cameroon',
'Canada': 'Canada',
'Cayman Islands': 'Cayman Islands',
'Cape Verde': 'Cape Verde',
'Chile': 'Chile',
'China': 'China',
'Colombia': 'Colombia',
'Congo (Democratic Republic of)': 'Congo (Democratic Republic of)',
'Congo (Republic of)': 'Congo (Republic of)',
'Cook Islands': 'Cook Islands',
'Costa Rica': 'Costa Rica',
"Côte d'Ivoire": "Côte d'Ivoire",
'Croatia': 'Croatia',
'Cuba': 'Cuba',
'Curacao': 'Curacao',
'Cyprus': 'Cyprus',
'Czech Republic': 'Czech Republic',
'Denmark': 'Denmark',
'Dominican Republic': 'Dominican Republic',
'Ecuador': 'Ecuador',
'Egypt': 'Egypt',
'El Salvador': 'El Salvador',
'Estonia': 'Estonia',
'Ethiopia': 'Ethiopia',
'Fiji': 'Fiji',
'Finland': 'Finland',
'France': 'France',
'Gabon': 'Gabon',
'Georgia': 'Georgia',
'Germany': 'Germany',
'Ghana': 'Ghana',
'Greece': 'Greece',
'Guatemala': 'Guatemala',
'Guernsey (States of)': 'Guernsey (States of)',
'Honduras': 'Honduras',
'Hong Kong': 'Hong Kong',
'Hungary': 'Hungary',
'Iceland': 'Iceland',
'India': 'India',
'Indonesia': 'Indonesia',
'Iraq': 'Iraq',
'Ireland': 'Ireland',
'Isle of Man': 'Isle of Man',
'Israel': 'Israel',
'Italy': 'Italy',
'Jamaica': 'Jamaica',
'Japan': 'Japan',
'Jersey (States of)': 'Jersey (States of)',
'Jordan': 'Jordan',
'Kazakhstan': 'Kazakhstan',
'Kenya': 'Kenya',
'Korea': 'Korea',
'Kuwait': 'Kuwait',
'Kyrgyzstan': 'Kyrgyzstan',
'Latvia': 'Latvia',
'Lebanon': 'Lebanon',
'Liechtenstein': 'Liechtenstein',
'Lithuania': 'Lithuania',
'Luxembourg': 'Luxembourg',
'Macao': 'Macao',
'Macedonia': 'Macedonia',
'Malaysia': 'Malaysia',
'Malta': 'Malta',
'Mauritius': 'Mauritius',
'Mexico': 'Mexico',
'Moldova': 'Moldova',
'Mongolia': 'Mongolia',
'Montenegro': 'Montenegro',
'Montserrat': 'Montserrat',
'Morocco': 'Morocco',
'Mozambique': 'Mozambique',
'Namibia': 'Namibia',
'Netherlands': 'Netherlands',
'New Zealand': 'New Zealand',
'Nicaragua': 'Nicaragua',
'Nigeria': 'Nigeria',
'Norway': 'Norway',
'Oman': 'Oman',
'Pakistan': 'Pakistan',
'Panama': 'Panama',
'Papua New Guinea': 'Papua New Guinea',
'Paraguay': 'Paraguay',
'Peru': 'Peru',
'Philippines': 'Philippines',
'Poland': 'Poland',
'Portugal': 'Portugal',
'Qatar': 'Qatar',
'Ras Al Khaimah (Emirate of)': 'Ras Al Khaimah (Emirate of)',
'Romania': 'Romania',
'Russia': 'Russia',
'Rwanda': 'Rwanda',
'Saudi Arabia': 'Saudi Arabia',
'Senegal': 'Senegal',
'Serbia': 'Serbia',
'Sharjah': 'Sharjah',
'Singapore': 'Singapore',
'Slovakia': 'Slovakia',
'Slovenia': 'Slovenia',
'South Africa': 'South Africa',
'Spain': 'Spain',
'Sri Lanka': 'Sri Lanka',
'St. Maarten': 'St. Maarten',
'St. Vincent & the Grenadines': 'St. Vincent & the Grenadines',
'Suriname': 'Suriname',
'Sweden': 'Sweden',
'Switzerland': 'Switzerland',
'Taiwan': 'Taiwan',
'Thailand': 'Thailand',
'Trinidad and Tobago': 'Trinidad and Tobago',
'Tunisia': 'Tunisia',
'Turkey': 'Turkey',
'Turks and Caicos Islands': 'Turks and Caicos Islands',
'Uganda': 'Uganda',
'Ukraine': 'Ukraine',
'United Arab Emirates': 'United Arab Emirates',
'United Kingdom': 'United Kingdom',
'United States': 'US',
'Uruguay': 'Uruguay',
'Venezuela': 'Venezuela',
'Vietnam': 'Vietnam',
'Zambia': 'Zambia',
'United States of America': 'US',
'Alderny': 'Guernsey (States of)',
'Andorra': 'Andorra (Principality of)',
'Bosnia and Herzogovina': 'Bosnia and Herzegovina',
'Eurozone': 'Eurozone',
'Fiji Islands': 'Fiji',
'Turkmenistan': 'Turkmenistan',
'Benin': 'Benin',
'Laos': 'Laos',
'Maldives': 'Maldives',
'Mali': 'Mali',
'Niger': 'Niger',
'Solomon Islands': 'Solomon Islands',
'Swaziland': 'Swaziland',
'Tajikistan': 'Tajikistan',
'Tanzania': 'Tanzania',
'Togo': 'Togo',
'Uzbekistan': 'Uzbekistan',
'Alderney': 'Guernsey (States of)',
'Bahamas - Off Shore Banking Center': 'Bahamas',
'Bahrain - Off Shore Banking Center': 'Bahrain',
'Cayman Islands - Off Shore Banking Center': 'Cayman Islands',
'Gibraltar': 'Gibraltar',
'Guernsey': 'Guernsey (States of)',
'Iran': 'Iran',
'Jersey': 'Jersey (States of)',
'Macau': 'Macao',
'Monaco': 'Monaco',
'Panama - Off Shore Banking Center': 'Panama',
'San Marino': 'San Marino',
'Sark': 'Guernsey (States of)',
'Trinidad & Tobago': 'Trinidad and Tobago',
'Alderney (Channel Islands)': 'Guernsey (States of)',
'Bahamas-Offshore': 'Bahamas',
'Bahrain-Offshore': 'Bahrain',
'Bosnia & Herzogovina': 'Bosnia and Herzegovina',
'Guernsey (Channel Islands)': 'Guernsey (States of)',
'Jersey (Channel Island)': 'Jersey (States of)',
'Panama-Offshore Banks': 'Panama',
'Papua New Guinea': 'Papua New Guinea',
'Sark (Channel Islands)': 'Guernsey (States of)',
'St. Vincent': 'St. Vincent & the Grenadines',
'Democratic Republic of Congo': 'Congo (Democratic Republic of)',
'Ras Al Kaminah': 'Ras Al Khaimah (Emirate of)',
'Republic of the Congo': 'Congo (Republic of)',
'Austria [1]': 'Austria',
'Belgium [1]': 'Belgium',
'Cyprus [1]': 'Cyprus',
'Finland [1]': 'Finland',
'France [1]': 'France',
'Germany [1]': 'Germany',
'Greece [1]': 'Greece',
'Ireland [1]': 'Ireland',
'Italy [1]': 'Italy',
'Luxembourg [1]': 'Luxembourg',
'Malta [1]': 'Malta',
'Netherlands [1]': 'Netherlands',
'Portugal [1]': 'Portugal',
'Slovenia [1]': 'Slovenia',
'Spain [1]': 'Spain',
'El Saklvador': 'El Salvador',
'Trinidad': 'Trinidad and Tobago',
'UK': 'United Kingdom',
'USA': 'US',
'United Arab Emirate': 'United Arab Emirates',
'Cayman islands': 'Cayman Islands',
'Costa': 'Costa Rica',
'Papua & New Guinea': 'Papua New Guinea',
'US': 'US',
'Nepal': 'Nepal',
'Bahamas-Off Shore Banking Center': 'Bahamas',
'Bahrain-Off Shore Banking': 'Bahrain',
'Cayman Islands Off Shore Banking': 'Cayman Islands',
'Panama Off-shore Banking': 'Panama'
}

In [19]:
# Rename 'Country' values in country_data using countries_unified_mapping
country_data['Country'] = country_data['Country'].map(countries_unified_mapping).fillna(country_data['Country'])

### Get WACC by country and industry

#### country_region_mapping

In [20]:
country_region_mapping = {
    'us':['US'],
    'china':['China'],
    'europe':[
        'Andorra (Principality of)',
        'Austria',
        'Belgium',
        'Cyprus',
        'Denmark',
        'Finland',
        'France',
        'Germany',
        'Greece',
        'Guernsey (States of)',
        'Gibraltar',
        'Iceland',
        'Ireland',
        'Isle of Man',
        'Italy',
        'Jersey (States of)',
        'Liechtenstein',
        'Monaco',
       'San Marino',
        'Luxembourg',
        'Malta',
        'Netherlands',
        'Norway',
        'Portugal',
        'Spain',
        'Sweden',
        'Switzerland',
        'Turkey',
        'United Kingdom',
    ],
    'india':['India'],
    'japan':['Japan'],
    'emerg':[
         'Angola',
         'Benin',
         'Botswana',
         'Burkina',
         'Faso',
         'Cameroon',
         'Cape Verde',
         'Congo (Democratic Republic of)',
         'Congo (Republic of)',
         "Côte d'Ivoire",
         'Egypt',
         'Ethiopia',
         'Gabon',
         'Ghana',
         'Kenya',
         'Mali',
         'Mauritius',
         'Morocco',
         'Mozambique',
         'Namibia',
         'Niger',
         'Nigeria',
         'Rwanda',
         'Senegal',
         'South Africa',
         'Swaziland',
         'Tanzania',
         'Togo',
         'Tunisia',
         'Uganda',
         'Zambia',
         'Bangladesh',
         'Cambodia',
         'China',
         'Fiji',
         'Hong Kong',
         'Indonesia',
        'Iran',
       'Papua  New Guinea',
        'Nepal',
         'Korea',
         'Laos',
         'Macao',
         'Malaysia',
         'Maldives',
         'Mongolia',
         'Pakistan',
         'Papua New Guinea',
         'Philippines',
         'Singapore',
         'Solomon Islands',
         'Sri Lanka',
         'Taiwan',
         'Thailand',
         'Vietnam',
         'Aruba',
 'Bahamas',
 'Barbados',
 'Bermuda',
 'Cayman Islands',
 'Cuba',
 'Curacao',
 'Dominican Republic',
 'Jamaica',
 'Montserrat',
 'St. Maarten',
 'St. Vincent & the Grenadines',
 'Trinidad and Tobago',
 'Turks and Caicos Islands',
            'Argentina',
 'Belize',
 'Bolivia',
 'Brazil',
 'Chile',
 'Colombia',
 'Costa Rica',
 'Ecuador',
 'El Salvador',
 'Guatemala',
 'Honduras',
 'Mexico',
 'Nicaragua',
 'Panama',
 'Paraguay',
 'Peru',
 'Suriname',
 'Uruguay',
 'Venezuela',
            'Albania',
 'Armenia',
 'Azerbaijan',
 'Belarus',
 'Bosnia and Herzegovina',
 'Bulgaria',
 'Croatia',
 'Czech Republic',
 'Estonia',
 'Georgia',
 'Hungary',
 'Kazakhstan',
 'Kyrgyzstan',
 'Latvia',
 'Lithuania',
 'Macedonia',
 'Moldova',
 'Montenegro',
 'Poland',
 'Romania',
 'Russia',
 'Serbia',
 'Slovakia',
 'Slovenia',
 'Tajikistan',
 'Turkmenistan',        
 'Ukraine',
 'Uzbekistan',
            'Abu Dhabi',
 'Bahrain',
 'Iraq',
 'Israel',
 'Jordan',
 'Kuwait',
 'Lebanon',
 'Oman',
 'Qatar',
 'Ras Al Khaimah (Emirate of)',
 'Saudi Arabia',
 'Sharjah',
 'United Arab Emirates'],
    'rest':['Australia',
            'Cook Islands',
            'New Zealand',
            'Canada',
           ]
}

In [21]:
# Convert country_region_mapping to a DataFrame
mapping_df = pd.DataFrame(
    [(country, region) for region, countries in country_region_mapping.items() for country in countries],
    columns=['Country', 'Region']
)

# Merge the mapping DataFrame with country_data to add the 'Region' column
country_data = country_data.merge(mapping_df, on='Country', how='left')

# Display the updated DataFrame
display(country_data)

,Country,Country Risk Premium,Region_x,Year,Region_y
0,Abu Dhabi,0.005998,Middle East,2017,emerg
1,Albania,0.054418,Eastern Europe & Russia,2017,emerg
2,Andorra (Principality of),0.026609,Western Europe,2017,europe
3,Angola,0.054418,Africa,2017,emerg
4,Argentina,0.078627,Central and South America,2017,emerg
...,...,...,...,...,...
3442,United Kingdom,0.000000,None,2004,europe
3443,US,0.000000,None,2004,us
3444,Uruguay,0.097500,None,2004,emerg
3445,Venezuela,0.067500,None,2004,emerg


In [22]:
country_data = country_data.dropna(subset='Region_y')

In [23]:
country_data = country_data.rename(columns={'Region_y': 'region'})

In [24]:
country_data = country_data.rename(columns={'Country Risk Premium': 'crp', 'Country':'country', 'Year':'year'})

### Get WACC by country and industry

In [25]:
# Perform an inner join on the 'region' column
wacc_by_country_industry = pd.merge(country_data, wacc_data, on=['region','year'], how='inner')

In [26]:
wacc_by_country_industry = wacc_by_country_industry.drop(columns=['Region_x'])

In [27]:
wacc_by_country_industry['share_of_equity'] = 1 / (1 + wacc_by_country_industry['d_e'])
wacc_by_country_industry['share_of_debt'] = wacc_by_country_industry['d_e'] / (1 + wacc_by_country_industry['d_e'])
wacc_by_country_industry['wacc'] = (wacc_by_country_industry['coe'] + wacc_by_country_industry['crp']) * wacc_by_country_industry['share_of_equity'] \
                                    + wacc_by_country_industry['cod'] * wacc_by_country_industry['share_of_debt'] * (1 - wacc_by_country_industry['tax_rate'])

In [28]:
wacc_by_country_industry = wacc_by_country_industry.drop(columns=['share_of_equity', 'share_of_debt'])

In [29]:
wacc_by_country_industry

,country,crp,year,region,industry,beta,coe,cod,tax_rate,d_e,wacc
0,Abu Dhabi,0.005998,2017,emerg,Advertising,1.309993,0.129430,0.0558,0.144928,0.127061,0.125540
1,Abu Dhabi,0.005998,2017,emerg,Aerospace/Defense,1.288743,0.127728,0.0538,0.118073,0.259296,0.115961
2,Abu Dhabi,0.005998,2017,emerg,Air Transport,0.910044,0.097394,0.0538,0.163303,1.160735,0.072032
3,Abu Dhabi,0.005998,2017,emerg,Apparel,0.792344,0.087967,0.0538,0.146877,0.264355,0.083915
4,Abu Dhabi,0.005998,2017,emerg,Auto & Truck,1.227810,0.122848,0.0538,0.134367,0.390738,0.105730
...,...,...,...,...,...,...,...,...,...,...,...
192110,US,0.000000,2004,us,Trucking,0.833333,0.082667,0.0575,0.273041,0.531801,0.068479
192111,US,0.000000,2004,us,Utility (Foreign),0.827500,0.082386,0.0525,0.167392,0.837816,0.064755
192112,US,0.000000,2004,us,Water Utility,0.565385,0.069752,0.0475,0.309724,0.662189,0.055026
192113,US,0.000000,2004,us,Wireless Networking,2.209615,0.149003,0.0625,0.051079,0.485069,0.119706


In [ ]:
# Fill missing 2025 with 2024 values
wacwacc_by_country_industry_2024 = wacc_by_country_industry[wacc_by_country_industry['year'] == 2024].copy()
wacc_by_country_industry_2024['year'] = 2025
wacc_by_country_industry = pd.concat([wacc_by_country_industry, wacc_by_country_industry_2024], ignore_index=True)
wacc_by_country_industry = wacc_by_country_industry.drop_duplicates(subset=['country_damodaran','industry_damodaran','year'], keep='first')

In [30]:
wacc_by_country_industry.to_parquet(work_dir+'/data/wacc/wacc_by_country_industry.parquet')

In [31]:
list(wacc_by_country_industry.industry.unique())

['Advertising',
 'Aerospace/Defense',
 'Air Transport',
 'Apparel',
 'Auto & Truck',
 'Auto Parts',
 'Bank (Money Center)',
 'Banks (Regional)',
 'Beverage (Alcoholic)',
 'Beverage (Soft)',
 'Broadcasting',
 'Brokerage & Investment Banking',
 'Building Materials',
 'Business & Consumer Services',
 'Cable TV',
 'Chemical (Basic)',
 'Chemical (Diversified)',
 'Chemical (Specialty)',
 'Coal & Related Energy',
 'Computer Services',
 'Computers/Peripherals',
 'Construction Supplies',
 'Diversified',
 'Drugs (Biotechnology)',
 'Drugs (Pharmaceutical)',
 'Education',
 'Electrical Equipment',
 'Electronics (Consumer & Office)',
 'Electronics (General)',
 'Engineering/Construction',
 'Entertainment',
 'Environmental & Waste Services',
 'Farming/Agriculture',
 'Financial Svcs. (Non-bank & Insurance)',
 'Food Processing',
 'Food Wholesalers',
 'Furn/Home Furnishings',
 'Green & Renewable Energy',
 'Healthcare Products',
 'Healthcare Support Services',
 'Heathcare Information and Technology',
 'Ho

In [32]:
list(wacc_by_country_industry.country.unique())

['Abu Dhabi',
 'Albania',
 'Andorra (Principality of)',
 'Angola',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Bermuda',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Bulgaria',
 'Burkina',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cayman Islands',
 'Cape Verde',
 'Chile',
 'China',
 'Colombia',
 'Congo (Democratic Republic of)',
 'Congo (Republic of)',
 'Cook Islands',
 'Costa Rica',
 "Côte d'Ivoire",
 'Croatia',
 'Cuba',
 'Curacao',
 'Cyprus',
 'Czech Republic',
 'Denmark',
 'Dominican Republic',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'Estonia',
 'Ethiopia',
 'Fiji',
 'Finland',
 'France',
 'Gabon',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Guatemala',
 'Guernsey (States of)',
 'Honduras',
 'Hong Kong',
 'Hungary',
 'Iceland',
 'India',
 'Indonesia',
 'Iraq',
 'Ireland',
 'Isle of Man',
 'Israel',
 'Italy',
 'Jamaica',
 'Japan',
 'Jersey (Stat

### WACC to ticker mapping

In [71]:
# companies_country = pd.read_csv(f'{work_dir}/data/frame_all_tickers_w_links_and_info.csv')
companies_country = pd.read_parquet(f'data/dec_25/fin_db_with_updated_ru.parquet') #combined pl, bs, cf, ratio sources

In [78]:
companies_country = companies_country[['ticker','exchange','exchange_name','exchange_country','exchange_code','country','industry','sector']]

In [84]:
companies_country = companies_country.drop_duplicates(subset=['ticker', 'exchange'])

In [85]:
# sector_industry_list
# companies_country.groupby(['industry','sector'])[['ticker']].agg('count').drop(columns='ticker').reset_index()
# list(companies_country.groupby(['sector','industry']).groups.keys())

In [86]:
industry_mapping = {
    # Academic & Educational Services
    ('Academic & Educational Services', 'Miscellaneous Educational Service Providers'): 'Education',
    ('Academic & Educational Services', 'School, College & University'): 'Education',

    # Communication Services
    ('Communication Services', 'Advertising Agencies'): 'Advertising',
    ('Communication Services', 'Broadcasting'): 'Broadcasting',
    ('Communication Services', 'Electronic Gaming & Multimedia'): 'Entertainment',
    ('Communication Services', 'Entertainment'): 'Entertainment',
    ('Communication Services', 'Internet Content & Information'): 'Information Services',
    ('Communication Services', 'Media & Entertainment'): 'Entertainment',
    ('Communication Services', 'Publishing'): 'Publishing & Newspapers',
    ('Communication Services', 'Telecom Services'): 'Telecom. Services',
    ('Communication Services', 'Telecommunications Services'): 'Telecom. Services',

    # Communications Equipment
    ('Communications Equipment', 'Information Technology'): 'Computer Services',  # Best fit

    # Conglomerates
    ('Conglomerates', 'Conglomerates'): 'Diversified',

    # Consumer Discretionary
    ('Consumer Discretionary', 'Telecommunications Services'): 'Telecom. Services',
    ('Consumer Discretionary', 'Apparel - Manufacturers'): 'Apparel',
    ('Consumer Discretionary', 'Apparel Manufacturing'): 'Apparel',
    ('Consumer Discretionary', 'Apparel Retail'): 'Retail (Special Lines)',
    ('Consumer Discretionary', 'Auto & Truck Dealerships'): 'Retail (Automotive)',
    ('Consumer Discretionary', 'Auto - Dealerships'): 'Retail (Automotive)',
    ('Consumer Discretionary', 'Auto - Recreational Vehicles'): 'Recreation',
    ('Consumer Discretionary', 'Auto Manufacturers'): 'Auto & Truck',
    ('Consumer Discretionary', 'Auto Parts'): 'Auto Parts',
    ('Consumer Discretionary', 'Automobiles & Auto Parts'): 'Auto & Truck',
    ('Consumer Discretionary', 'Consumer Electronics'): 'Electronics (Consumer & Office)',
    ('Consumer Discretionary', 'Department Stores'): 'Retail (General)',
    ('Consumer Discretionary', 'Diversified Retail'): 'Retail (General)',
    ('Consumer Discretionary', 'Footwear & Accessories'): 'Shoe',
    ('Consumer Discretionary', 'Furnishings, Fixtures & Appliances'): 'Furn/Home Furnishings',
    ('Consumer Discretionary', 'Gambling'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Gambling, Resorts & Casinos'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Home Improvement'): 'Building Materials',
    ('Consumer Discretionary', 'Home Improvement Retail'): 'Retail (Building Supply)',
    ('Consumer Discretionary', 'Homebuilding & Construction Supplies'): 'Homebuilding',
    ('Consumer Discretionary', 'Hotels & Entertainment Services'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Hotels, Restaurants & Leisure'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Household Goods'): 'Household Products',
    ('Consumer Discretionary', 'Internet Retail'): 'Retail (Online)',
    ('Consumer Discretionary', 'Leisure'): 'Recreation',
    ('Consumer Discretionary', 'Leisure Products'): 'Recreation',
    ('Consumer Discretionary', 'Lodging'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Luxury Goods'): 'Household Products',  # or 'Personal Products' if exists; closest match
    ('Consumer Discretionary', 'Manufacturing - Textiles'): 'Textile',
    ('Consumer Discretionary', 'Media & Publishing'): 'Publishing & Newspapers',
    ('Consumer Discretionary', 'Packaging & Containers'): 'Packaging & Container',
    ('Consumer Discretionary', 'Personal Products & Services'): 'Household Products',
    ('Consumer Discretionary', 'Personal Services'): 'Business & Consumer Services',
    ('Consumer Discretionary', 'Recreational Vehicles'): 'Recreation',
    ('Consumer Discretionary', 'Residential Construction'): 'Homebuilding',
    ('Consumer Discretionary', 'Resorts & Casinos'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Restaurants'): 'Restaurant/Dining',
    ('Consumer Discretionary', 'Specialty Retail'): 'Retail (Special Lines)',
    ('Consumer Discretionary', 'Specialty Retailers'): 'Retail (Special Lines)',
    ('Consumer Discretionary', 'Textile Manufacturing'): 'Textile',
    ('Consumer Discretionary', 'Textiles & Apparel'): 'Apparel',
    ('Consumer Discretionary', 'Textiles, Apparel &Â\xa0Luxury Goods'): 'Apparel',
    ('Consumer Discretionary', 'Travel Lodging'): 'Hotel/Gaming',
    ('Consumer Discretionary', 'Travel Services'): 'Transportation',

    # Consumer Goods
    ('Consumer Goods', 'Beverages - Brewers'): 'Beverage (Alcoholic)',
    ('Consumer Goods', 'Electronic Equipment'): 'Electronics (General)',
    ('Consumer Goods', 'Food - Major Diversified'): 'Food Processing',
    ('Consumer Goods', 'Personal Products'): 'Household Products',
    ('Consumer Goods', 'Recreational Goods, Other'): 'Recreation',

    # Consumer Non-Cyclicals
    ('Consumer Non-Cyclicals', 'Beverages'): 'Beverage (Soft)',
    ('Consumer Non-Cyclicals', 'Consumer Goods Conglomerates'): 'Diversified',
    ('Consumer Non-Cyclicals', 'Food & Drug Retailing'): 'Retail (Grocery and Food)',
    ('Consumer Non-Cyclicals', 'Food & Tobacco'): 'Food Processing',
    ('Consumer Non-Cyclicals', 'Personal & Household Products & Services'): 'Household Products',

    # Consumer Staples
    ('Consumer Staples', ' Beverages - Brewers'): 'Beverage (Alcoholic)',  # note leading space
    ('Consumer Staples', ' Beverages - Non-Alcoholic'): 'Beverage (Soft)',
    ('Consumer Staples', ' Beverages - Wineries & Distilleries'): 'Beverage (Alcoholic)',
    ('Consumer Staples', 'Agricultural Farm Products'): 'Farming/Agriculture',
    ('Consumer Staples', 'Agricultural Inputs'): 'Farming/Agriculture',
    ('Consumer Staples', 'Beverages'): 'Beverage (Soft)',
    ('Consumer Staples', 'Beverages - Alcoholic'): 'Beverage (Alcoholic)',
    ('Consumer Staples', 'Beverages - Brewers'): 'Beverage (Alcoholic)',
    ('Consumer Staples', 'Beverages - Non-Alcoholic'): 'Beverage (Soft)',
    ('Consumer Staples', 'Beverages - Wineries & Distilleries'): 'Beverage (Alcoholic)',
    ('Consumer Staples', 'Confectioners'): 'Food Processing',
    ('Consumer Staples', 'Discount Stores'): 'Retail (General)',
    ('Consumer Staples', 'Education & Training Services'): 'Education',
    ('Consumer Staples', 'Farm Products'): 'Farming/Agriculture',
    ('Consumer Staples', 'Food Confectioners'): 'Food Processing',
    ('Consumer Staples', 'Food Distribution'): 'Food Wholesalers',
    ('Consumer Staples', 'Food Products'): 'Food Processing',
    ('Consumer Staples', 'Grocery Stores'): 'Retail (Grocery and Food)',
    ('Consumer Staples', 'Household & Personal Products'): 'Household Products',
    ('Consumer Staples', 'Packaged Foods'): 'Food Processing',
    ('Consumer Staples', 'Tobacco'): 'Tobacco',

    # Energy
    ('Energy', 'Coal'): 'Coal & Related Energy',
    ('Energy', 'Oil & Gas'): 'Oil/Gas (Integrated)',
    ('Energy', 'Oil & Gas Drilling'): 'Oil/Gas (Production and Exploration)',
    ('Energy', 'Oil & Gas Energy'): 'Oil/Gas (Integrated)',
    ('Energy', 'Oil & Gas Equipment & Services'): 'Oilfield Svcs/Equip.',
    ('Energy', 'Oil & Gas Exploration & Production'): 'Oil/Gas (Production and Exploration)',
    ('Energy', 'Oil & Gas Integrated'): 'Oil/Gas (Integrated)',
    ('Energy', 'Oil & Gas Midstream'): 'Oil/Gas Distribution',
    ('Energy', 'Oil & Gas Refining & Marketing'): 'Oil/Gas (Integrated)',
    ('Energy', 'Oil & Gas Related Equipment and Services'): 'Oilfield Svcs/Equip.',
    ('Energy', 'Oil, Gas & Consumable Fuels'): 'Oil/Gas (Integrated)',
    ('Energy', 'Renewable Energy'): 'Green & Renewable Energy',
    ('Energy', 'Solar'): 'Green & Renewable Energy',
    ('Energy', 'Thermal Coal'): 'Coal & Related Energy',
    ('Energy', 'Uranium'): 'Metals & Mining',

    # Financials
    ('Financials', ' Banks - Regional'): 'Banks (Regional)',
    ('Financials', ' Insurance - Diversified'): 'Insurance (General)',
    ('Financials', ' Insurance - Life'): 'Insurance (Life)',
    ('Financials', ' Insurance - Property & Casualty'): 'Insurance (Prop/Cas.)',
    ('Financials', 'Asset Management'): 'Investments & Asset Management',
    ('Financials', 'Asset Management - Bonds'): 'Investments & Asset Management',
    ('Financials', 'Asset Management - Global'): 'Investments & Asset Management',
    ('Financials', 'Asset Management - Income'): 'Investments & Asset Management',
    ('Financials', 'Banking Services'): 'Brokerage & Investment Banking',
    ('Financials', 'Banks'): 'Bank (Money Center)',
    ('Financials', 'Banks - Diversified'): 'Bank (Money Center)',
    ('Financials', 'Banks - Regional'): 'Banks (Regional)',
    ('Financials', 'Capital Markets'): 'Brokerage & Investment Banking',
    ('Financials', 'Consumer Finance'): 'Financial Svcs. (Non-bank & Insurance)',
    ('Financials', 'Credit Services'): 'Financial Svcs. (Non-bank & Insurance)',
    ('Financials', 'Diversified Financial Services'): 'Financial Svcs. (Non-bank & Insurance)',
    ('Financials', 'Financial - Capital Markets'): 'Brokerage & Investment Banking',
    ('Financials', 'Financial - Credit Services'): 'Financial Svcs. (Non-bank & Insurance)',
    ('Financials', 'Financial - Diversified'): 'Financial Svcs. (Non-bank & Insurance)',
    ('Financials', 'Financial - Mortgages'): 'Mortgage Finance',
    ('Financials', 'Financial Conglomerates'): 'Investments & Asset Management',
    ('Financials', 'Financial Data & Stock Exchanges'): 'Brokerage & Investment Banking',
    ('Financials', 'Insurance'): 'Insurance (General)',
    ('Financials', 'Insurance - Diversified'): 'Insurance (General)',
    ('Financials', 'Insurance - Life'): 'Insurance (Life)',
    ('Financials', 'Insurance - Property & Casualty'): 'Insurance (Prop/Cas.)',
    ('Financials', 'Insurance - Reinsurance'): 'Reinsurance',
    ('Financials', 'Insurance - Specialty'): 'Insurance (Prop/Cas.)',
    ('Financials', 'Insurance Brokers'): 'Brokerage & Investment Banking',
    ('Financials', 'Insurance-Reinsurance'): 'Reinsurance',
    ('Financials', 'Investment - Banking & Investment Services'): 'Brokerage & Investment Banking',
    ('Financials', 'Investment Banking & Investment Services'): 'Brokerage & Investment Banking',
    ('Financials', 'Investment Holding Companies'): 'Investments & Asset Management',
    ('Financials', 'Mortgage Finance'): 'Mortgage Finance',
    ('Financials', 'Property & Casualty Insurance'): 'Insurance (Prop/Cas.)',
    ('Financials', 'Real Estate - Development'): 'Real Estate (Development)',
    ('Financials', 'Rental & Leasing Services'): 'Real Estate (Operations & Services)',
    ('Financials', 'Shell Companies'): 'Unclassified',
    ('Healthcare', ' Drug Manufacturers - General'): 'Drugs (Pharmaceutical)',
    ('Healthcare', ' Drug Manufacturers - Specialty & Generic'): 'Drugs (Pharmaceutical)',
    ('Healthcare', 'Biotechnology'): 'Drugs (Biotechnology)',
    ('Industrials', 'Biotechnology'): 'Drugs (Biotechnology)',
    ('Healthcare', 'Biotechnology & Medical Research'): 'Drugs (Biotechnology)',
    ('Healthcare', 'Diagnostics & Research'): 'Healthcare Support Services',
    ('Healthcare', 'Drug Manufacturers - General'): 'Drugs (Pharmaceutical)',
    ('Healthcare', 'Drug Manufacturers - Specialty & Generic'): 'Drugs (Pharmaceutical)',
    ('Healthcare', 'Education & Training Services'): 'Education',
    ('Healthcare', 'Health Information Services'): 'Heathcare Information and Technology',
    ('Healthcare', 'Healthcare Plans'): 'Healthcare Support Services',
    ('Healthcare', 'Healthcare Providers & Services'): 'Hospitals/Healthcare Facilities',
    ('Healthcare', 'Life Sciences Tools & Services'): 'Healthcare Support Services',
    ('Healthcare', 'Medical - Diagnostics & Research'): 'Healthcare Support Services',
    ('Healthcare', 'Medical - Equipment & Services'): 'Healthcare Products',
    ('Healthcare', 'Medical - Healthcare Information Services'): 'Heathcare Information and Technology',
    ('Healthcare', 'Medical - Healthcare Plans'): 'Healthcare Support Services',
    ('Healthcare', 'Medical - Pharmaceuticals'): 'Drugs (Pharmaceutical)',
    ('Healthcare', 'Medical - Specialties'): 'Healthcare Products',
    ('Healthcare', 'Medical Care Facilities'): 'Hospitals/Healthcare Facilities',
    ('Healthcare', 'Medical Devices'): 'Healthcare Products',
    ('Healthcare', 'Medical Distribution'): 'Healthcare Support Services',
    ('Healthcare', 'Medical Instruments & Supplies'): 'Healthcare Products',
    ('Healthcare', 'Pharmaceutical Retailers'): 'Retail (General)',
    ('Healthcare', 'Pharmaceuticals'): 'Drugs (Pharmaceutical)',
    ('Industrial Goods', 'Diversified Machinery'): 'Machinery',
    ('Industrial Goods', 'Farm & Construction Machinery'): 'Machinery',
    ('Industrial Goods', 'Residential Construction'): 'Homebuilding',
    ('Industrial Goods', 'Waste Management'): 'Environmental & Waste Services',
    ('Industrials', 'Aerospace & Defense'): 'Aerospace/Defense',
    ('Industrials', 'Agricultural - Machinery'): 'Machinery',
    ('Industrials', 'Airlines'): 'Air Transport',
    ('Industrials', 'Airports & Air Services'): 'Air Transport',
    ('Industrials', 'Auto Parts'): 'Auto Parts',
    ('Industrials', 'Building Materials'): 'Building Materials',
    ('Industrials', 'Building Products & Equipment'): 'Building Materials',
    ('Industrials', 'Business Equipment & Supplies'): 'Office Equipment & Services',
    ('Industrials', 'Conglomerates'): 'Diversified',
    ('Industrials', 'Construction'): 'Engineering/Construction',
    ('Industrials', 'Construction\xa0& Engineering'): 'Engineering/Construction',
    ('Industrials', 'Consulting Services'): 'Business & Consumer Services',
    ('Industrials', 'Diversified Industrial Goods Wholesale'): 'Retail (Distributors)',
    ('Industrials', 'Education & Training Services'): 'Education',
    ('Industrials', 'Electrical Equipment & Parts'): 'Electrical Equipment',
    ('Industrials', 'Engineering & Construction'): 'Engineering/Construction',
    ('Industrials', 'Environmental Services'): 'Environmental & Waste Services',
    ('Industrials', 'Farm & Heavy Construction Machinery'): 'Machinery',
    ('Industrials', 'Freight & Logistics Services'): 'Transportation',
    ('Industrials', 'General Transportation'): 'Transportation',
    ('Industrials', 'Industrial - Capital Goods'): 'Machinery',
    ('Industrials', 'Industrial - Infrastructure Operations'): 'Infrastructure Operations',
    ('Industrials', 'Industrial - Machinery'): 'Machinery',
    ('Industrials', 'Industrial - Pollution & Treatment Controls'): 'Environmental & Waste Services',
    ('Industrials', 'Industrial - Specialties'): 'Machinery',
    ('Industrials', 'Industrial Distribution'): 'Retail (Distributors)',
    ('Industrials', 'Industrial Materials'): 'Metals & Mining',
    ('Industrials', 'Infrastructure Operations'): 'Engineering/Construction',
    ('Industrials', 'Integrated Freight & Logistics'): 'Transportation',
    ('Industrials', 'Machinery'): 'Machinery',
    ('Industrials', 'Machinery, Tools, Heavy Vehicles, Trains & Ships'): 'Machinery',
    ('Industrials', 'Manufacturing - Metal Fabrication'): 'Metal Fabricating',
    ('Industrials', 'Manufacturing - Miscellaneous'): 'Diversified',
    ('Industrials', 'Manufacturing - Textiles'): 'Textile',
    ('Industrials', 'Manufacturing - Tools & Accessories'): 'Tools & Accessories',
    ('Industrials', 'Marine Shipping'): 'Shipbuilding & Marine',
    ('Industrials', 'Medical Distribution'): 'Healthcare Support Services',
    ('Industrials', 'Metal Fabrication'): 'Metal Fabricating',
    ('Industrials', 'Passenger Transportation Services'): 'Transportation',
    ('Industrials', 'Pollution & Treatment Controls'): 'Environmental & Waste Services',
    ('Industrials', 'Professional & Commercial Services'): 'Business & Consumer Services',
    ('Industrials', 'Railroads'): 'Transportation (Railroads)',
    ('Industrials', 'Rental & Leasing Services'): 'Real Estate (Operations & Services)',
    ('Industrials', 'Security & Protection Services'): 'Business & Consumer Services',
    ('Industrials', 'Software - Application'): 'Software (System & Application)',
    ('Industrials', 'Specialty Business Services'): 'Business & Consumer Services',
    ('Industrials', 'Specialty Industrial Machinery'): 'Machinery',
    ('Industrials', 'Staffing & Employment Services'): 'Business & Consumer Services',
    ('Industrials', 'Technology Distributors'): 'Retail (Distributors)',
    ('Industrials', 'Tools & Accessories'): 'Tools & Accessories',
    ('Industrials', 'Trading Companies & Distributors'): 'Retail (Distributors)',
    ('Industrials', 'Transport Infrastructure'): 'Engineering/Construction',
    ('Industrials', 'Trucking'): 'Trucking',
    ('Industrials', 'Waste Management'): 'Environmental & Waste Services',
    ('Leisure Products', 'Consumer Discretionary'): 'Retail (General)',
    ('Materials', 'Agricultural - Commodities/Milling'): 'Farming/Agriculture',
    ('Materials', 'Agricultural Inputs'): 'Farming/Agriculture',
    ('Materials', 'Aluminum'): 'Metals & Mining',
    ('Materials', 'Building Materials'): 'Building Materials',
    ('Materials', 'Chemicals'): 'Chemical (Basic)',
    ('Materials', 'Coking Coal'): 'Coal & Related Energy',
    ('Materials', 'Containers & Packaging'): 'Packaging & Container',
    ('Materials', 'Copper'): 'Metals & Mining',
    ('Materials', 'Gold'): 'Precious Metals',
    ('Materials', 'Industrial - Specialties'): 'Chemical (Specialty)',
    ('Materials', 'Industrial Materials'): 'Metals & Mining',
    ('Materials', 'Lumber & Wood Production'): 'Paper/Forest Products',
    ('Materials', 'Metals & Mining'): 'Metals & Mining',
    ('Materials', 'Other Industrial Metals & Mining'): 'Metals & Mining',
    ('Materials', 'Other Precious Metals'): 'Precious Metals',
    ('Materials', 'Other Precious Metals & Mining'): 'Precious Metals',
    ('Materials', 'Paper & Forest Products'): 'Paper/Forest Products',
    ('Materials', 'Paper & Paper Products'): 'Paper/Forest Products',
    ('Materials', 'Paper, Lumber & Forest Products'): 'Paper/Forest Products',
    ('Materials', 'Silver'): 'Precious Metals',
    ('Materials', 'Specialty Chemicals'): 'Chemical (Specialty)',
    ('Materials', 'Steel'): 'Steel',
    ('Media', 'Communication Services'): 'Broadcasting',
    ('Metals & Mining', 'Materials'): 'Metals & Mining',
    ('Real Estate', ' REIT - Diversified'): 'R.E.I.T.',
    ('Real Estate', ' REIT - Industrial'): 'R.E.I.T.',
    ('Real Estate', ' REIT - Mortgage'): 'R.E.I.T.',
    ('Real Estate', ' Real Estate - Development'): 'Real Estate (Development)',
    ('Real Estate', 'Equity Real Estate Investment Trusts (REITs)'): 'R.E.I.T.',
    ('Real Estate', 'Land Subdividers and Developers, Except Cemeteries'): 'Real Estate (Development)',
    ('Real Estate', 'REIT - Diversified'): 'R.E.I.T.',
    ('Real Estate', 'REIT - Healthcare Facilities'): 'Hospitals/Healthcare Facilities',
    ('Real Estate', 'REIT - Hotel & Motel'): 'Hotel/Gaming',
    ('Real Estate', 'REIT - Industrial'): 'R.E.I.T.',
    ('Real Estate', 'REIT - Mortgage'): 'R.E.I.T.',
    ('Real Estate', 'REIT - Office'): 'R.E.I.T.',
    ('Real Estate', 'REIT - Residential'): 'R.E.I.T.',
    ('Real Estate', 'REIT - Retail'): 'R.E.I.T.',
    ('Real Estate', 'REIT - Specialty'): 'R.E.I.T.',
    ('Real Estate', 'REIT-Hotel & Motel'): 'Hotel/Gaming',
    ('Real Estate', 'REIT-Office'): 'R.E.I.T.',
    ('Real Estate', 'REIT-Residential'): 'R.E.I.T.',
    ('Real Estate', 'Real Estate'): 'Real Estate (General/Diversified)',
    ('Real Estate', 'Real Estate - Development'): 'Real Estate (Development)',
    ('Real Estate', 'Real Estate - Diversified'): 'Real Estate (General/Diversified)',
    ('Real Estate', 'Real Estate - General'): 'Real Estate (General/Diversified)',
    ('Real Estate', 'Real Estate Management & Development'): 'Real Estate (Development)',
    ('Real Estate', 'Real Estate Operations'): 'Real Estate (Operations & Services)',
    ('Real Estate', 'Real Estate Services'): 'Real Estate (Operations & Services)',
    ('Real Estate', 'Residential & Commercial REITs'): 'R.E.I.T.',
    ('Services', 'Auto Dealerships'): 'Retail (Automotive)',
    ('Services', 'Electronics Wholesale'): 'Retail (Distributors)',
    ('Services', 'Jewelry Stores'): 'Retail (Special Lines)',
    ('Technology', ' Software - Application'): 'Software (System & Application)',
    ('Technology', ' Software - Infrastructure'): 'Software (System & Application)',
    ('Technology', 'Aerospace & Defense'): 'Aerospace/Defense',
    ('Technology', 'Communication Equipment'): 'Telecom. Equipment',
    ('Technology', 'Communications & Networking'): 'Telecom. Equipment',
    ('Technology', 'Computer Hardware'): 'Computers/Peripherals',
    ('Technology', 'Computers, Phones & Household Electronics'): 'Electronics (Consumer & Office)',
    ('Technology', 'Consulting Services'): 'Business & Consumer Services',
    ('Technology', 'Consumer Electronics'): 'Electronics (Consumer & Office)',
    ('Technology', 'Education & Training Services'): 'Education',
    ('Technology', 'Electronic Components'): 'Electronics (General)',
    ('Technology', 'Electronic Equipment & Parts'): 'Electronics (General)',
    ('Technology', 'Electronic Equipment, Instruments & Components'): 'Electronics (General)',
    ('Technology', 'Electronic Gaming & Multimedia'): 'Entertainment',
    ('Technology', 'Electronics & Computer Distribution'): 'Retail (Distributors)',
    ('Technology', 'Hardware, Equipment & Parts'): 'Electronics (General)',
    ('Technology', 'IT Services'): 'Computer Services',
    ('Technology', 'Information Technology Services'): 'Computer Services',
    ('Technology', 'Internet Content & Information'): 'Information Services',
    ('Technology', 'Media & Entertainment'): 'Entertainment',
    ('Technology', 'Scientific & Technical Instrument'): 'Precision Instrument',
    ('Technology', 'Scientific & Technical Instruments'): 'Precision Instrument',
    ('Technology', 'Semiconductor Equipment & Materials'): 'Semiconductor Equip',
    ('Technology', 'Semiconductors'): 'Semiconductor',
    ('Technology', 'Semiconductors & Semiconductor Equipment'): 'Semiconductor',
    ('Technology', 'Software - Application'): 'Software (System & Application)',
    ('Technology', 'Software - Infrastructure'): 'Software (System & Application)',
    ('Technology', 'Software - Services'): 'Software (System & Application)',
    ('Technology', 'Solar'): 'Green & Renewable Energy',
    ('Technology', 'Specialty Retail'): 'Retail (Special Lines)',
    ('Technology', 'Technology Distributors'): 'Retail (Distributors)',
    ('Technology', 'Telecommunications Services'): 'Telecom. Services',
    ('Textiles, Apparel &Â\xa0Luxury Goods', 'Consumer Discretionary'): 'Apparel',
    ('Utilities', 'Utilities - Diversified'): 'Utility (General)',
    ('Utilities', ' Utilities - Diversified'): 'Utility (General)',
    ('Utilities', 'Utilities - Independent Power Producers'): 'Power',
    ('Utilities', ' Utilities - Independent Power Producers'): 'Power',
    ('Utilities', 'Utilities - Regulated Gas'): 'Natural Gas (Distrib.)',
    ('Utilities', 'Utilities - Regulated Water'): 'Utility (Water)',
    ('Utilities', ' Utilities - Regulated Gas'): 'Natural Gas (Distrib.)',
    ('Utilities', ' Utilities - Regulated Water'): 'Utility (Water)',
    ('Utilities', 'Utilities - Renewable'): 'Green & Renewable Energy',
    ('Utilities', ' Utilities - Renewable'): 'Green & Renewable Energy',
    ('Utilities', 'Electric Utilities'): 'Power',
    ('Utilities', 'Electrical Utilities & IPPs'): 'Power',
    ('Utilities', 'General Utilities'): 'Utility (General)',
    ('Utilities', 'Independent Power Producers'): 'Power',
    ('Utilities', 'Natural Gas Utilities'): 'Natural Gas (Distrib.)',
    ('Utilities', 'Regulated Electric'): 'Power',
    ('Utilities', 'Regulated Gas'): 'Natural Gas (Distrib.)',
    ('Utilities', 'Regulated Water'): 'Utility (Water)',
    ('Utilities', 'Utilities - Diversified'): 'Utility (General)',
    ('Utilities', ' Utilities - Diversified'): 'Utility (General)',
    ('Utilities', 'Utilities - Independent Power Producers'): 'Power',
    ('Utilities', 'Utilities - Regulated Electric'): 'Power',
    ('Utilities', 'Utilities - Regulated Gas'): 'Natural Gas (Distrib.)',
    ('Utilities', 'Utilities - Regulated Water'): 'Utility (Water)',
    ('Utilities', 'Utilities-Regulated Electric'): 'Power',
    ('Utilities', 'Water & Related Utilities'): 'Utility (Water)',
    (None, 'Abrasive, Asbestos, And Miscellaneous Nonmetallic Mineral Products'): 'Total Market',
    (None, 'Accident and Health Insurance'): 'Insurance (Prop/Cas.)',
    (None, 'Accounting, Auditing, and Bookkeeping Services'): 'Business & Consumer Services',
    (None, 'Adhesives and Sealants'): 'Chemical (Specialty)',
    (None, 'Advertising'): 'Advertising',
    (None, 'Advertising Agencies'): 'Advertising',
    (None, 'Agricultural Chemicals'): 'Chemical (Basic)',
    (None, 'Agricultural Production Crops'): 'Farming/Agriculture',
    (None, 'Agricultural Services'): 'Farming/Agriculture',
    (None, 'Agriculture Production Livestock and Animal Specialties'): 'Farming/Agriculture',
    (None, 'Air Courier Services'): 'Transportation',
    (None, 'Air Transportation, Nonscheduled'): 'Air Transport',
    (None, 'Air Transportation, Scheduled'): 'Air Transport',
    (None,
     'Air-Conditioning and Warm Air Heating Equipment and Commercial and Industrial Refrigeration Equipment'): 'Electrical Equipment',
    (None, 'Aircraft'): 'Aerospace/Defense',
    (None, 'Aircraft And Parts'): 'Aerospace/Defense',
    (None, 'Aircraft Engines and Engine Parts'): 'Aerospace/Defense',
    (None, 'Aircraft Parts and Auxiliary Equipment, Not Elsewhere Classified'): 'Aerospace/Defense',
    (None, 'Airports, Flying Fields, and Airport Terminal Services'): 'Air Transport',
    (None, 'Amusement Parks'): 'Entertainment',
    (None, 'Amusement and Recreation Services'): 'Recreation',
    (None, 'Amusement and Recreation Services, Not Elsewhere Classified'): 'Recreation',
    (None, 'Apparel and Accessory Stores'): 'Apparel',
    (float('nan'), 'Apparel and Other Finished Products Made From Fabrics and Similar Materials'): 'Apparel',
    (float('nan'), 'Apparel, Piece Goods, And Notions'): 'Apparel',
    (float('nan'), 'Arrangement of Transportation of Freight and Cargo'): 'Transportation',
    (float('nan'), 'Asphalt Paving And Roofing Materials'): 'Building Materials',
    (float('nan'), 'Auto and Home Supply Stores'): 'Retail (Building Supply)',
    (float('nan'), 'Automatic Controls for Regulating Residential and Commercial Environments and Appliances'): 'Electronics (Consumer & Office)',
    (float('nan'), 'Automotive Dealers and Gasoline Service Stations'): 'Retail (Automotive)',
    (float('nan'), 'Automotive Rental And Leasing, Without Drivers'): 'Auto & Truck',
    (float('nan'), 'Automotive Repair, Services, and Parking'): 'Auto Parts',
    (float('nan'), 'Bakery Products'): 'Food Processing',
    (float('nan'), 'Ball and Roller Bearings'): 'Machinery',
    (float('nan'), 'Beer, Wine, And Distilled Alcoholic Beverages'): 'Beverage (Alcoholic)',
    (float('nan'), 'Beverages'): 'Beverage (Soft)',
    (float('nan'), 'Biological Products, Except Diagnostic Substances'): 'Drugs (Biotechnology)',
    (float('nan'), 'Bituminous Coal And Lignite Mining'): 'Coal & Related Energy',
    (float('nan'), 'Bolts, Nuts, Screws, Rivets, and Washers'): 'Machinery',
    (float('nan'), 'Book Printing'): 'Publishing & Newspapers',
    (float('nan'), 'Books: Publishing, or Publishing and Printing'): 'Publishing & Newspapers',
    (float('nan'), 'Bottled and Canned Soft Drinks and Carbonated Waters'): 'Beverage (Soft)',
    (float('nan'), 'Bread and Other Bakery Products, Except Cookies and Crackers'): 'Food Processing',
    (float('nan'), 'Broadwoven Fabric Mills, Cotton'): 'Apparel',
    (float('nan'), 'Broadwoven Fabric Mills, Manmade Fiber and Silk'): 'Apparel',
    (float('nan'), 'Building Construction General Contractors and Operative Builders'): 'Engineering/Construction',
    (float('nan'), 'Building Materials'): 'Building Materials',
    (float('nan'), 'Building Materials, Hardware, Garden Supply, and Mobile Home Dealers'): 'Building Materials',
    (float('nan'), 'Business Consulting Services, Not Elsewhere Classified'): 'Business & Consumer Services',
    (float('nan'), 'Business Credit Institutions'): 'Banks (Regional)',
    (float('nan'), 'Business Development Company'): 'Investments & Asset Management',
    (float('nan'), 'Business Services, not elsewhere classified'): 'Business & Consumer Services',
    (float('nan'), 'Cable and Other Pay Television Services'): 'Cable TV',
    (float('nan'), 'Calculating and Accounting Machines, Except Electronic Computers'): 'Office Equipment & Services',
    (float('nan'), 'Canned Fruits, Vegetables, Preserves, Jams, and Jellies'): 'Food Processing',
    (float('nan'), 'Canned, Frozen, And Preserved Fruits, Vegetables, and Food Specialties'): 'Food Processing',
    (float('nan'), 'Carpets and Rugs'): 'Furn/Home Furnishings',
    (float('nan'), 'Catalog and Mail-Order Houses'): 'Retail (General)',
    (float('nan'), 'Cement, Hydraulic'): 'Building Materials',
    (float('nan'), 'Chemicals And Allied Products'): 'Chemical (Diversified)',
    (float('nan'), 'Chemicals and Allied Products'): 'Chemical (Diversified)',
    (float('nan'), 'Chemicals and Chemical Preparations, Not Elsewhere Classified'): 'Chemical (Specialty)',
    (float('nan'), 'Child Day Care Services'): 'Education',
    (float('nan'), 'Chocolate and Cocoa Products'): 'Food Processing',
    (float('nan'), 'Cigarettes'): 'Tobacco',
    (float('nan'), 'Coal Mining'): 'Coal & Related Energy',
    (float('nan'), 'Coating, Engraving, And Allied Services'): 'Engineering/Construction',
    (float('nan'), 'Cogeneration Services and Small Power Producers'): 'Power',
    (float('nan'), 'Commercial Banks'): 'Bank (Money Center)',
    (float('nan'), 'Commercial Physical and Biological Research'): 'Drugs (Biotechnology)',
    (float('nan'), 'Commercial Printing'): 'Publishing & Newspapers',
    (float('nan'), 'Commercial Sports'): 'Entertainment',
    (float('nan'), 'Communications Equipment, not elsewhere classified'): 'Telecom. Equipment',
    (float('nan'), 'Communications Services, Not Elsewhere Classified'): 'Telecom. Services',
    (float('nan'), 'Computer And Office Equipment'): 'Computers/Peripherals',
    (float('nan'), 'Computer Communications Equipment'): 'Telecom. Equipment',
    (float('nan'), 'Computer Integrated Systems Design'): 'Software (System & Application)',
    (float('nan'), 'Computer Peripheral Equipment, not elsewhere classified'): 'Computers/Peripherals',
    (float('nan'), 'Computer Processing and Data Preparation and Processing Services'): 'Computer Services',
    (float('nan'), 'Computer Programming Services'): 'Software (System & Application)',
    (float('nan'), 'Computer Programming, Data Processing, And Other Computer Related Services'): 'Computer Services',
    (float('nan'), 'Computer Related Services, not elsewhere classified'): 'Computer Services',
    (float('nan'), 'Computer Rental and Leasing'): 'Computer Services',
    (float('nan'), 'Computer Storage Devices'): 'Computers/Peripherals',
    (float('nan'), 'Computer and Computer Software Stores'): 'Computer Services',
    (float('nan'), 'Computers and Computer Peripheral Equipment and Software'): 'Computers/Peripherals',
    (float('nan'), 'Concrete Products, Except Block and Brick'): 'Building Materials',
    (float('nan'), 'Concrete Work'): 'Engineering/Construction',
    (float('nan'), 'Concrete, Gypsum, And Plaster Products'): 'Building Materials',
    (float('nan'), 'Conglomerates'): 'Diversified',
    (float('nan'), 'Construction Machinery and Equipment'): 'Machinery',
    (float('nan'), 'Construction Special Trade Contractors'): 'Engineering/Construction',
    (float('nan'), 'Construction and Mining (Except Petroleum) Machinery and Equipment'): 'Machinery',
    (float('nan'), 'Construction, Mining, And Materials Handling'): 'Machinery',
    (float('nan'), 'Consumer Credit Reporting Agencies, Mercantile'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Convenience Stores'): 'Retail (Grocery and Food)',
    (float('nan'), 'Converted Paper And Paperboard Products, Except Containers And Boxes'): 'Paper/Forest Products',
    (float('nan'), 'Cookies and Crackers'): 'Food Processing',
    (float('nan'), 'Copper Ores'): 'Metals & Mining',
    (float('nan'), 'Costume Jewelry, Costume Novelties, Buttons, And Miscellaneous Notions, Except Precious Metal'): 'Apparel',
    (float('nan'), 'Credit Reporting Services'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Crude Petroleum and Natural Gas'): 'Oil/Gas (Production and Exploration)',
    (float('nan'), 'Cut Stone and Stone Products'): 'Building Materials',
    (float('nan'), 'Cutlery, Handtools, And General Hardware'): 'Building Materials',
    (float('nan'), "Cutting Tools, Machine Tool Accessories, and Machinists' Precision Measuring Devices"): 'Machinery',
    (float('nan'), 'Dairy Products'): 'Food Processing',
    (float('nan'), 'Deep Sea Foreign Transportation of Freight'): 'Transportation',
    (float('nan'), 'Dental Equipment and Supplies'): 'Healthcare Products',
    (float('nan'), 'Department Stores'): 'Retail (General)',
    (float('nan'), 'Detective, Guard, and Armored Car Services'): 'Business & Consumer Services',
    (float('nan'), 'Distilled and Blended Liquors'): 'Beverage (Alcoholic)',
    (float('nan'), 'Diversified Multi-Media'): 'Entertainment',
    (float('nan'), 'Division F: Wholesale Trade'): 'Retail (Distributors)',
    (float('nan'), 'Dolls and Stuffed Toys'): 'Recreation',
    (float('nan'), 'Drawing and Insulating of Nonferrous Wire'): 'Electrical Work',
    (float('nan'), 'Drilling Oil and Gas Wells'): 'Oil/Gas (Production and Exploration)',
    (float('nan'), 'Drug Stores and Proprietary Stores'): 'Retail (General)',
    (float('nan'), "Drugs, Drug Proprietaries, and Druggists' Sundries"): 'Drugs (Pharmaceutical)',
    (float('nan'), 'Durable Goods, not elsewhere classified'): 'Diversified',
    (float('nan'), 'Eating And Drinking Places'): 'Restaurant/Dining',
    (float('nan'), 'Eating Places'): 'Restaurant/Dining',
    (float('nan'), 'Educational Services'): 'Education',
    (float('nan'), 'Electric Housewares and Fans'): 'Electronics (Consumer & Office)',
    (float('nan'), 'Electric Lighting And Wiring Equipment'): 'Electrical Equipment',
    (float('nan'), 'Electric Services'): 'Power',
    (float('nan'), 'Electric and Other Services Combined'): 'Power',
    (float('nan'), 'Electric, Gas, and Sanitary Services'): 'Utility (General)',
    (float('nan'), 'Electrical Apparatus and Equipment Wiring Supplies, and Construction Materials'): 'Electrical Equipment',
    (float('nan'), 'Electrical Appliances, Television and Radio Sets'): 'Electronics (Consumer & Office)',
    (float('nan'), 'Electrical Industrial Apparatus'): 'Electrical Equipment',
    (float('nan'), 'Electrical Work'): 'Engineering/Construction',
    (float('nan'), 'Electromedical and Electrotherapeutic Apparatus'): 'Healthcare Products',
    (float('nan'), 'Electronic Coils, Transformers, and Other Inductors'): 'Electronics (General)',
    (float('nan'), 'Electronic Components And Accessories'): 'Semiconductor',
    (float('nan'), 'Electronic Components, not elsewhere classified'): 'Electronics (General)',
    (float('nan'), 'Electronic Computers'): 'Computers/Peripherals',
    (float('nan'), 'Electronic Connectors'): 'Semiconductor',
    (float('nan'), 'Electronic Parts and Equipment, not elsewhere classified'): 'Electronics (General)',
    (float('nan'), 'Electronic and Other Electrical Equipment and Components, except Computer Equipment'): 'Electronics (General)',
    (float('nan'), 'Employment Agencies'): 'Business & Consumer Services',
    (float('nan'), 'Engineering Services'): 'Engineering/Construction',
    (float('nan'), 'Engineering, Accounting, Research, Management, and Related Services'): 'Business & Consumer Services',
    (float('nan'), 'Engines And Turbines'): 'Machinery',
    (float('nan'), 'Equipment Rental and Leasing, not elsewhere classified'): 'Business & Consumer Services',
    (float('nan'), 'Fabricated Pipe and Pipe Fittings'): 'Steel',
    (float('nan'), 'Fabricated Plate Work (Boiler Shops)'): 'Steel',
    (float('nan'), 'Fabricated Rubber Products, Not Elsewhere Classified'): 'Rubber& Tires',
    (float('nan'), 'Fabricated Structural Metal'): 'Steel',
    (float('nan'), 'Fabricated Structural Metal Products'): 'Steel',
    (float('nan'), 'Facilities Support Management Services'): 'Business & Consumer Services',
    (float('nan'), 'Family Clothing Stores'): 'Retail (General)',
    (float('nan'), 'Farm Machinery and Equipment'): 'Machinery',
    (float('nan'), 'Farm Supplies'): 'Agriculture',
    (float('nan'), 'Farm-Product Raw Materials, not elsewhere classified'): 'Farming/Agriculture',
    (float('nan'), 'Farm-product Raw Materials'): 'Farming/Agriculture',
    (float('nan'), 'Fats And Oils'): 'Food Processing',
    (float('nan'), 'Federal and Federally-Sponsored Credit Agencies'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Finance Lessors'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Finance services'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Fire, Marine, and Casualty Insurance'): 'Insurance (Prop/Cas.)',
    (float('nan'), 'Fishing, Hunting, and Trapping'): 'Farming/Agriculture',
    (float('nan'), 'Flat Glass'): 'Building Materials',
    (float('nan'), 'Food Stores'): 'Retail (Grocery and Food)',
    (float('nan'), 'Food and Kindred Products'): 'Food Processing',
    (float('nan'), 'Footwear, Except Rubber'): 'Shoe',
    (float('nan'), 'Forestry'): 'Paper/Forest Products',
    (float('nan'), 'Functions Related to Depository Banking, Not Elsewhere Classified'): 'Bank (Money Center)',
    (float('nan'), 'Furniture And Home Furnishings'): 'Furn/Home Furnishings',
    (float('nan'), 'Furniture Stores'): 'Retail (General)',
    (float('nan'), 'Furniture and Fixtures, not elsewhere classified'): 'Furn/Home Furnishings',
    (float('nan'), "Games, Toys, and Children's Vehicles, Except Dolls and Bicycles"): 'Recreation',
    (float('nan'), 'Gas and Other Services Combined'): 'Utility (General)',
    (float('nan'), 'Gaskets, Packing, And Sealing Devices And Rubber And Plastics Hose And Belting'): 'Rubber& Tires',
    (float('nan'), 'General Building Contractors-nonresidential'): 'Engineering/Construction',
    (float('nan'), 'General Building Contractors-residential'): 'Engineering/Construction',
    (float('nan'), 'General Contractors-Nonresidential Buildings, Other than Industrial Buildings and Warehouses'): 'Engineering/Construction',
    (float('nan'), 'General Contractors-Residential Buildings, Other Than Single-Family'): 'Engineering/Construction',
    (float('nan'), 'General Contractors-Single-Family Houses'): 'Homebuilding',
    (float('nan'), 'General Industrial Machinery And Equipment'): 'Machinery',
    (float('nan'), 'General Industrial Machinery and Equipment, Not Elsewhere'): 'Machinery',
    (float('nan'), 'General Medical and Surgical Hospitals'): 'Hospitals/Healthcare Facilities',
    (float('nan'), 'Glass And Glassware, Pressed Or Blown'): 'Building Materials',
    (float('nan'), 'Glass Containers'): 'Packaging & Container',
    (float('nan'), 'Glass Products, Made of Purchased Glass'): 'Building Materials',
    (float('nan'), 'Gold And Silver Ores'): 'Precious Metals',
    (float('nan'), 'Gold Ores'): 'Precious Metals',
    (float('nan'), 'Grain Mill Products'): 'Food Processing',
    (float('nan'), 'Groceries And Related Products'): 'Retail (Grocery and Food)',
    (float('nan'), 'Groceries and Related Products, not elsewhere classified'): 'Retail (Grocery and Food)',
    (float('nan'), 'Groceries, General Line'): 'Retail (Grocery and Food)',
    (float('nan'), 'Grocery Stores'): 'Retail (Grocery and Food)',
    (float('nan'), 'Guided Missiles And Space Vehicles And Parts'): 'Aerospace/Defense',
    (float('nan'), 'Hardware'): 'Electronics (General)',
    (float('nan'), 'Hardware, And Plumbing And Heating Equipment'): 'Building Materials',
    (float('nan'), 'Hazardous Waste Management'): 'Environmental & Waste Services',
    (float('nan'), 'Health Services'): 'Healthcare Support Services',
    (float('nan'), 'Health and Allied Services, not elsewhere classified'): 'Healthcare Support Services',
    (float('nan'), 'Heating Equipment, Except Electric And Warm Air; And Plumbing Fixtures'): 'Building Materials',
    (float('nan'), 'Heating Equipment, Except Electric and Warm Air Furnaces'): 'Building Materials',
    (float('nan'), 'Heavy Construction Other Than Building Construction Contractors'): 'Engineering/Construction',
    (float('nan'), 'Help Supply Services'): 'Business & Consumer Services',
    (float('nan'), 'Highway and Street Construction, Except Elevated Highways'): 'Engineering/Construction',
    (float('nan'), 'Hobby, Toy, and Game Shops'): 'Recreation',
    (float('nan'), 'Home Furniture, Furnishings, and Equipment Stores'): 'Furn/Home Furnishings',
    (float('nan'), 'Home Health Care Services'): 'Healthcare Support Services',
    (float('nan'), 'Hospital and Medical Service Plans'): 'Healthcare Support Services',
    (float('nan'), 'Hospitals'): 'Hospitals/Healthcare Facilities',
    (float('nan'), 'Hotels and Motels'): 'Hotel/Gaming',
    (float('nan'), 'Hotels, Rooming Houses, Camps, and Other Lodging Places'): 'Hotel/Gaming',
    (float('nan'), 'Household Appliances'): 'Electronics (Consumer & Office)',
    (float('nan'), 'Household Audio and Video Equipment'): 'Electronics (Consumer & Office)',
    (float('nan'), 'Household Furniture'): 'Furn/Home Furnishings',
    (float('nan'), 'Ice Cream and Frozen Desserts'): 'Food Processing',
    (float('nan'), 'In Vitro and In Vivo Diagnostic Substances'): 'Healthcare Products',
    (float('nan'), 'Industrial Gases'): 'Chemical (Basic)',
    (float('nan'), 'Industrial Inorganic Chemicals'): 'Chemical (Basic)',
    (float('nan'), 'Industrial Inorganic Chemicals, not elsewhere classified'): 'Chemical (Basic)',
    (float('nan'), 'Industrial Instruments for Measurement, Display, and Control of Process Variables; and Related Products'): 'Electronics (General)',
    (float('nan'), 'Industrial Machinery and Equipment'): 'Machinery',
    (float('nan'), 'Industrial Organic Chemicals'): 'Chemical (Specialty)',
    (float('nan'), 'Industrial Process Furnaces and Ovens'): 'Machinery',
    (float('nan'), 'Industrial Sand'): 'Metals & Mining',
    (float('nan'), 'Industrial Supplies'): 'Business & Consumer Services',
    (float('nan'), 'Industrial Trucks, Tractors, Trailers, and Stackers'): 'Auto & Truck',
    (float('nan'), 'Industrial and Commercial Fans and Blowers and Air Purification Equipment'): 'Machinery',
    (float('nan'), 'Industrial and Commercial Machinery and Computer Equipment'): 'Machinery',
    (float('nan'), 'Information Retrieval Services'): 'Information Services',
    (float('nan'), 'Instruments for Measuring and Testing of Electricity and Electrical Signals'): 'Electronics (General)',
    (float('nan'), 'Insurance Agents, Brokers, and Service'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Insurance Carriers'): 'Insurance (General)',
    (float('nan'), 'Insurance Carriers, not elsewhere classified'): 'Insurance (General)',
    (float('nan'), 'Investment Advice'): 'Investments & Asset Management',
    (float('nan'), 'Investment Offices'): 'Investments & Asset Management',
    (float('nan'), 'Investors, not elsewhere classified'): 'Investments & Asset Management',
    (float('nan'), 'Iron And Steel Foundries'): 'Steel',
    (float('nan'), 'Jewelry Stores'): 'Retail (Special Lines)',
    (float('nan'), 'Jewelry, Precious Metal'): 'Precious Metals',
    (float('nan'), 'Jewelry, Silverware, And Plated Ware'): 'Retail (Special Lines)',
    (float('nan'), 'Jewelry, Watches, Precious Stones, and Precious Metals'): 'Retail (Special Lines)',
    (float('nan'), 'Knitting Mills'): 'Apparel',
    (float('nan'), 'Laboratory Analytical Instruments'): 'Healthcare Products',
    (float('nan'), 'Laboratory Apparatus and Furniture'): 'Healthcare Products',
    (float('nan'), 'Land Subdividers And Developers'): 'Real Estate (Development)',
    (float('nan'), 'Land Subdividers and Developers, Except Cemeteries'): 'Real Estate (Development)',
    (float('nan'), 'Lawn and Garden Tractors and Home Lawn and Garden Equipment'): 'Machinery',
    (float('nan'), 'Leather and Leather Products'): 'Apparel',
    (float('nan'), 'Legal Services'): 'Business & Consumer Services',
    (float('nan'), 'Lessors of Real Property, Not Elsewhere Classified'): 'Real Estate (Operations & Services)',
    (float('nan'), 'Life Insurance'): 'Insurance (Life)',
    (float('nan'), 'Loan Brokers'): 'Financial Svcs. (Non-bank & Insurance)',
    (float('nan'), 'Local and Suburban Transit'): 'Transportation',
    (float('nan'), 'Local and Suburban Transit and Interurban Highway Passenger Transportation'): 'Transportation',
    (float('nan'), 'Lumber And Other Construction Materials'): 'Building Materials',
    (float('nan'), 'Lumber and Other Building Materials Dealers'): 'Building Materials',
    (float('nan'), 'Lumber and Wood Products, except Furniture'): 'Paper/Forest Products',
    (float('nan'), 'Lumber, Plywood, Millwork, and Wood Panels'): 'Building Materials',
    (float('nan'), 'Machine Tools, Metal Cutting Types'): 'Machinery',
    (float('nan'), 'Machinery, Equipment, And Supplies'): 'Machinery',
    (float('nan'), 'Mailing, Reproduction, Commercial Art And Photography, and Stenographic Services'): 'Business & Consumer Services',
    (float('nan'), 'Malt Beverages'): 'Beverage (Alcoholic)',
    (float('nan'), 'Management Consulting Services'): 'Business & Consumer Services',
    (float('nan'), 'Management Investment Offices, Open-End'): 'Investments & Asset Management',
    (float('nan'), 'Management Services'): 'Business & Consumer Services',
    (float('nan'), 'Manifold Business Forms'): 'Business & Consumer Services',
    (float('nan'), 'Measuring and Controlling Devices, not elsewhere classified'): 'Electronics (General)',
    (float('nan'), 'Meat Packing Plants'): 'Food Processing',
    (float('nan'), 'Mechanical Power Transmission Equipment, Not Elsewhere Classified'): 'Machinery',
    (float('nan'), 'Medical Laboratories'): 'Healthcare Support Services',
    (float('nan'), 'Medical, Dental, and Hospital Equipment and Supplies'): 'Healthcare Products',

    (np.nan, 'Medicinal Chemicals and Botanical Products'): 'Drugs (Biotechnology)',
    (np.nan, "Men's And Boys' Furnishings, Work Clothing, And Allied Garments"): 'Apparel',
    (np.nan, 'Metal Cans'): 'Packaging & Container',
    (np.nan, 'Metal Doors, Sash, Frames, Molding, and Trim Manufacturing'): 'Building Materials',
    (np.nan, 'Metal Forgings And Stampings'): 'Machinery',
    (np.nan, 'Metal Mining'): 'Metals & Mining',
    (np.nan, 'Metal Mining Services'): 'Oilfield Svcs/Equip.',
    (np.nan, 'Metal Shipping Barrels, Drums, Kegs, and Pails'): 'Packaging & Container',
    (np.nan, 'Metals And Minerals, Except Petroleum'): 'Metals & Mining',
    (np.nan, 'Metals Service Centers and Offices'): 'Metals & Mining',
    (np.nan, 'Metalworking Machinery And Equipment'): 'Machinery',
    (np.nan, 'Millwork, Veneer, Plywood, And Structural Wood Members'): 'Building Materials',
    (np.nan, 'Mineral Royalty Traders'): 'Precious Metals',
    (np.nan, 'Mining Machinery and Equipment, Except Oil and Gas Field Machinery and Equipment'): 'Machinery',
    (np.nan, 'Mining and Quarrying Of Nonmetallic Minerals, except Fuels'): 'Metals & Mining',
    (np.nan, 'Miscellaneous Amusement And Recreation'): 'Recreation',
    (np.nan, 'Miscellaneous Business Services'): 'Business & Consumer Services',
    (np.nan, 'Miscellaneous Chemical Products'): 'Chemical (Specialty)',
    (np.nan, 'Miscellaneous Durable Goods'): 'Diversified',
    (np.nan, 'Miscellaneous Electrical Machinery, Equipment, and Supplies'): 'Electrical Equipment',
    (np.nan, 'Miscellaneous Equipment Rental And Leasing'): 'Business & Consumer Services',
    (np.nan, 'Miscellaneous Fabricated Metal Products'): 'Machinery',
    (np.nan, 'Miscellaneous Fabricated Textile Products'): 'Apparel',
    (np.nan, 'Miscellaneous Food Preparations And Kindred Products'): 'Food Processing',
    (np.nan, 'Miscellaneous Furniture And Fixtures'): 'Furn/Home Furnishings',
    (np.nan, 'Miscellaneous General Merchandise Stores'): 'Retail (General)',
    (np.nan, 'Miscellaneous Health And Allied Services, Not Elsewhere Classified'): 'Healthcare Support Services',
    (np.nan, 'Miscellaneous Industrial And Commercial Machinery And Equipment'): 'Machinery',
    (np.nan, 'Miscellaneous Manufacturing Industries'): 'Diversified',
    (np.nan, 'Miscellaneous Metal Ores'): 'Metals & Mining',
    (np.nan, 'Miscellaneous Metal Ores, Not Elsewhere Classified'): 'Metals & Mining',
    (np.nan, 'Miscellaneous Non-durable Goods'): 'Diversified',
    (np.nan, 'Miscellaneous Personal Services, Not Elsewhere Classified'): 'Business & Consumer Services',
    (np.nan, 'Miscellaneous Plastics Products'): 'Chemical (Specialty)',
    (np.nan, 'Miscellaneous Primary Metal Products'): 'Metals & Mining',
    (np.nan, 'Miscellaneous Products Of Petroleum And Coal'): 'Oil/Gas (Integrated)',
    (np.nan, 'Miscellaneous Publishing'): 'Publshing & Newspapers',
    (np.nan, 'Miscellaneous Repair Services'): 'Business & Consumer Services',
    (np.nan, 'Miscellaneous Retail'): 'Retail (General)',
    (np.nan, 'Miscellaneous Services'): 'Business & Consumer Services',
    (np.nan, 'Miscellaneous Shopping Goods Stores'): 'Retail (General)',
    (np.nan, 'Miscellaneous Structural Metal Work'): 'Engineering/Construction',
    (np.nan, 'Miscellaneous Transportation Equipment'): 'Transportation',
    (np.nan, 'Miscellaneous business Credit Institutions'): 'Financial Svcs. (Non-bank & Insurance)',
    (np.nan, 'Mortgage Bankers and Loan Correspondents'): 'Financial Svcs. (Non-bank & Insurance)',
    (np.nan, 'Motion Picture Theaters'): 'Entertainment',
    (np.nan, 'Motion Picture and Video Tape Distribution'): 'Entertainment',
    (np.nan, 'Motion Picture and Video Tape Production'): 'Entertainment',
    (np.nan, 'Motor Freight Transportation and Warehousing'): 'Transportation',
    (np.nan, 'Motor Homes'): 'Auto & Truck',
    (np.nan, 'Motor Vehicle Dealers (Used Only)'): 'Retail (Automotive)',
    (np.nan, 'Motor Vehicle Parts and Accessories'): 'Auto Parts',
    (np.nan, 'Motor Vehicle Supplies and New Parts'): 'Auto Parts',
    (np.nan, 'Motor Vehicles And Motor Vehicle Parts And Supplies'): 'Auto & Truck',
    (np.nan, 'Motor Vehicles and Passenger Car Bodies'): 'Auto & Truck',
    (np.nan, 'Motorcycles, Bicycles, and Parts'): 'Auto Parts',
    (np.nan, 'Motors and Generators'): 'Electrical Equipment',
    (np.nan, 'Multi-Sector Holdings'): 'Diversified',
    (np.nan, 'Museums, Art Galleries, and Botanical and Zoological Gardens'): 'Entertainment',
    (np.nan, 'Natural Gas Distribution'): 'Oil/Gas Distribution',
    (np.nan, 'Natural Gas Transmission'): 'Oil/Gas Distribution',
    (np.nan, 'Natural Gas Transmission and Distribution'): 'Oil/Gas Distribution',
    (np.nan, 'Newspapers: Publishing, or Publishing and Printing'): 'Publshing & Newspapers',
    (np.nan, 'Non-Depository Credit Institutions'): 'Financial Svcs. (Non-bank & Insurance)',
    (np.nan, 'Non-operating Establishments'): 'Diversified',
    (np.nan, 'Nonferrous Foundries (castings)'): 'Metals & Mining',
    (np.nan, 'Nonstore Retailers'): 'Retail (Online)',
    (np.nan, 'Nursing And Personal Care Facilities'): 'Hospitals/Healthcare Facilities',
    (np.nan, 'Office Furniture'): 'Furn/Home Furnishings',
    (np.nan, 'Office Furniture, except Wood'): 'Furn/Home Furnishings',
    (np.nan, 'Office Machines, not elsewhere classified'): 'Computers/Peripherals',
    (np.nan, 'Offices and Clinics of Doctors of Medicine'): 'Hospitals/Healthcare Facilities',
    (np.nan, 'Offices of Bank Holding Companies'): 'Bank (Money Center)',
    (np.nan, 'Offices of Holding Companies, Not Elsewhere Classified'): 'Diversified',
    (np.nan, 'Oil Royalty Traders'): 'Precious Metals',
    (np.nan, 'Oil and Gas Extraction'): 'Oil/Gas (Production and Exploration)',
    (np.nan, 'Oil and Gas Field Exploration Services'): 'Oilfield Svcs/Equip.',
    (np.nan, 'Oil and Gas Field Machinery and Equipment'): 'Oilfield Svcs/Equip.',
    (np.nan, 'Oil and Gas Field Services, not elsewhere classified'): 'Oilfield Svcs/Equip.',
    (np.nan, 'Operative Builders'): 'Homebuilding',
    (np.nan, 'Operators of Apartment Buildings'): 'Real Estate (Operations & Services)',
    (np.nan, 'Operators of Nonresidential Buildings'): 'Real Estate (Operations & Services)',
    (np.nan, 'Ophthalmic Goods'): 'Healthcare Products',
    (np.nan, 'Optical Instruments and Lenses'): 'Healthcare Products',
    (np.nan, 'Ordnance And Accessories, Except Vehicles And Guided Missiles'): 'Aerospace/Defense',
    (np.nan, 'Orthopedic, Prosthetic, and Surgical Appliances and Supplies'): 'Healthcare Products',
    (np.nan, 'Other'): 'Unclassified',
    (np.nan, 'Paints, Varnishes, Lacquers, Enamels, and Allied Products'): 'Chemical (Specialty)',
    (np.nan, 'Paper And Paper Products'): 'Paper/Forest Products',
    (np.nan, 'Paper Mills'): 'Paper/Forest Products',
    (np.nan, 'Paper and Allied Products'): 'Paper/Forest Products',
    (np.nan, 'Paperboard Containers And Boxes'): 'Packaging & Container',
    (np.nan, 'Paperboard Mills'): 'Paper/Forest Products',
    (np.nan, 'Partitions, Shelving, Lockers, And Office And Store Fixtures'): 'Furn/Home Furnishings',
    (np.nan, 'Patent Owners and Lessors'): 'Investments & Asset Management',
    (np.nan, 'Pens, Pencils, And Other Artists Materials'): 'Office Equipment & Services',
    (np.nan, 'Perfumes, Cosmetics, and Other Toilet Preparations'): 'Household Products',
    (np.nan, 'Periodicals: Publishing, or Publishing and Printing'): 'Publshing & Newspapers',
    (np.nan, 'Personal Credit Institutions'): 'Financial Svcs. (Non-bank & Insurance)',
    (np.nan, 'Personal Services'): 'Business & Consumer Services',
    (np.nan, 'Petroleum Bulk Stations and Terminals'): 'Oil/Gas Distribution',
    (np.nan, 'Petroleum Refining'): 'Oil/Gas (Integrated)',
    (np.nan, 'Petroleum and Petroleum Products Wholesalers, Except Bulk Stations and Terminals'): 'Oil/Gas Distribution',
    (np.nan, 'Pharmaceutical Preparations'): 'Drugs (Pharmaceutical)',
    (np.nan, 'Phonograph Records and Prerecorded Audio Tapes and Disks'): 'Entertainment',
    (np.nan, 'Photofinishing Laboratories'): 'Business & Consumer Services',
    (np.nan, 'Photographic Equipment and Supplies'): 'Electronics (General)',
    (np.nan, 'Pipelines, Except Natural Gas'): 'Oil/Gas Distribution',
    (np.nan, 'Plastics Bottles'): 'Packaging & Container',
    (np.nan, 'Plastics Foam Products'): 'Chemical (Specialty)',
    (np.nan, 'Plastics Materials And Synthetic Resins, Synthetic Rubber, Cellulosic And Other Manmade Fibers, Except Glass'): 'Chemical (Specialty)',
    (np.nan, 'Plastics Materials, Synthetic Resins, and Nonvulcanizable Elastomers'): 'Chemical (Specialty)',
    (np.nan, 'Plastics Products, not elsewhere classified'): 'Chemical (Specialty)',
    (np.nan, 'Plastics, Foil, and Coated Paper Bags'): 'Packaging & Container',
    (np.nan, 'Pottery And Related Products'): 'Furn/Home Furnishings',
    (np.nan, 'Poultry Slaughtering and Processing'): 'Food Processing',
    (np.nan, 'Power, Distribution, and Specialty Transformers'): 'Electrical Equipment',
    (np.nan, 'Prefabricated Metal Buildings and Components'): 'Building Materials',
    (np.nan, 'Prefabricated Wood Buildings and Components'): 'Building Materials',
    (np.nan, 'Prepackaged Software'): 'Software (System & Application)',
    (np.nan, 'Prepared Fresh or Frozen Fish and Seafoods'): 'Food Processing',
    (np.nan, 'Primary Metal Industries'): 'Metals & Mining',
    (np.nan, 'Primary Production of Aluminum'): 'Metals & Mining',
    (np.nan, 'Primary Smelting And Refining Of Nonferrous Metals'): 'Metals & Mining',
    (np.nan, 'Printed Circuit Boards'): 'Semiconductor Equip',
    (np.nan, 'Printing Trades Machinery and Equipment'): 'Machinery',
    (np.nan, 'Printing, Publishing, and Allied Industries'): 'Publshing & Newspapers',
    (np.nan, 'Professional And Commercial Equipment And Supplies'): 'Office Equipment & Services',
    (np.nan, 'Professional Sports Clubs and Promoters'): 'Entertainment',
    (np.nan, 'Public Building and Related Furniture'): 'Furn/Home Furnishings',
    (np.nan, 'Public Warehousing And Storage'): 'Transportation',
    (np.nan, 'Pulp Mills'): 'Paper/Forest Products',
    (np.nan, 'Pumps and Pumping Equipment'): 'Machinery',
    (np.nan, 'Racing, Including Track Operation'): 'Recreation',
    (np.nan, 'Radio Broadcasting Stations'): 'Broadcasting',
    (np.nan, 'Radio and Television Broadcasting and Communications Equipment'): 'Electronics (General)',
    (np.nan, 'Radio, Television, and Consumer Electronics Stores'): 'Retail (General)',
    (np.nan, 'Radiotelephone Communications'): 'Telecom. Services',
    (np.nan, 'Railroad Equipment'): 'Transportation (Railroads)',
    (np.nan, 'Railroads, Line-Haul Operating'): 'Transportation (Railroads)',
    (np.nan, 'Real Estate'): 'Real Estate (General/Diversified)',
    (np.nan, 'Real Estate Agents and Managers'): 'Real Estate (Operations & Services)',
    (np.nan, 'Real Estate Investment Trusts'): 'R.E.I.T.',
    (np.nan, 'Real Estate Operators (except Developers) And Lessors'): 'Real Estate (Operations & Services)',
    (np.nan, 'Recreational Vehicle Parks and Campsites'): 'Hotel/Gaming',
    (np.nan, 'Refrigeration And Service Industry Machinery'): 'Machinery',
    (np.nan, 'Refuse Systems'): 'Environmental & Waste Services',
    (np.nan, 'Retail Stores, Not Elsewhere Classified'): 'Retail (General)',
    (np.nan, 'Rolling, Drawing, And Extruding Of Nonferrous Metals'): 'Metals & Mining',
    (np.nan, 'Rubber and Plastics Footwear'): 'Shoe',
    (np.nan, 'Sanitary Services'): 'Environmental & Waste Services',
    (np.nan, 'Sausages and Other Prepared Meat Products'): 'Food Processing',
    (np.nan, 'Savings Institutions, Federally Chartered'): 'Banks (Regional)',
    (np.nan, 'Savings Institutions, Not Federally Chartered'): 'Banks (Regional)',
    (np.nan, 'Sawmills and Planing Mills, General'): 'Paper/Forest Products',
    (np.nan, 'Scrap and Waste Materials'): 'Environmental & Waste Services',

    (float('nan'), 'Screw Machine Products'): 'Machinery',
    (float('nan'), 'Search, Detection, Navigation, Guidance, Aeronautical, and Nautical Systems and Instruments'): 'Electronics (General)',
    (float('nan'), 'Search, Detection, Navigation, Guidance, Aeronautical, and Nautical Systems, Instruments, and Equipment'): 'Electronics (General)',
    (float('nan'), 'Secondary Smelting and Refining of Nonferrous Metals'): 'Metals & Mining',
    (float('nan'), 'Security Brokers, Dealers, and Flotation Companies'): 'Brokerage & Investment Banking',
    (float('nan'), 'Security and Commodity Brokers, Dealers, Exchanges, and Services'): 'Brokerage & Investment Banking',
    (float('nan'), 'Semiconductors and Related Devices'): 'Semiconductor',
    (float('nan'), 'Services Allied to Motion Picture Distribution'): 'Entertainment',
    (float('nan'), 'Services Allied to Motion Picture Production'): 'Entertainment',
    (float('nan'), 'Services To Dwellings And Other Buildings'): 'Real Estate (Operations & Services)',
    (float('nan'), 'Services, not elsewhere classified'): 'Other',
    (float('nan'), 'Sheet Metal Work'): 'Construction Supplies',
    (float('nan'), 'Ship And Boat Building And Repairing'): 'Shipbuilding & Marine',
    (float('nan'), 'Shoe Stores'): 'Shoe',
    (float('nan'), 'Short-Term Business Credit Institutions, Except Agricultural'): 'Banks (Regional)',
    (float('nan'), 'Shortening, Table Oils, Margarine, and Other Edible Fats and Oils, Not Elsewhere Classified'): 'Food Processing',
    (float('nan'), 'Silver Ores'): 'Precious Metals',
    (float('nan'), 'Skilled Nursing Care Facilities'): 'Hospitals/Healthcare Facilities',
    (float('nan'), 'Soap, Detergents, And Cleaning Preparations; Perfumes, Cosmetics, and Other Toilet Preparations'): 'Household Products',
    (float('nan'), 'Social Services'): 'Business & Consumer Services',
    (float('nan'), 'Special Industry Machinery, Except Metalworking'): 'Machinery',
    (float('nan'), 'Special Industry Machinery, not elsewhere classified'): 'Machinery',
    (float('nan'), 'Specialty Cleaning, Polishing, and Sanitation Preparations'): 'Household Products',
    (float('nan'), 'Specialty Outpatient Facilities, Not Elsewhere Classified'): 'Hospitals/Healthcare Facilities',
    (float('nan'), 'Sporting and Athletic Goods, not elsewhere classified'): 'Apparel',
    (float('nan'), 'State Commercial Banks'): 'Banks (Regional)',
    (float('nan'), 'Steel Pipe and Tubes'): 'Steel',
    (float('nan'), 'Steel Works, Blast Furnaces (Including Coke Ovens), and Rolling Mills'): 'Steel',
    (float('nan'), 'Steel Works, Blast Furnaces, And Rolling And Finishing Mills'): 'Steel',
    (float('nan'), 'Structural Clay Products'): 'Building Materials',
    (float('nan'), 'Sugar And Confectionery Products'): 'Food Processing',
    (float('nan'), 'Sugarcane and Sugar Beets'): 'Farming/Agriculture',
    (float('nan'), 'Surety Insurance'): 'Insurance (Prop/Cas.)',
    (float('nan'), 'Surgical and Medical Instruments and Apparatus'): 'Healthcare Products',
    (float('nan'), 'Surgical, Medical, And Dental Instruments And Supplies'): 'Healthcare Products',
    (float('nan'), 'Switchgear and Switchboard Apparatus'): 'Electrical Equipment',
    (float('nan'), 'Telephone Communications'): 'Telecom. Services',
    (float('nan'), 'Telephone Communications, Except Radiotelephone'): 'Telecom. Services',
    (float('nan'), 'Telephone Interconnect Systems'): 'Telecom. Equipment',
    (float('nan'), 'Telephone and Telegraph Apparatus'): 'Telecom. Equipment',
    (float('nan'), 'Television Broadcasting Stations'): 'Broadcasting',
    (float('nan'), 'Testing Laboratories'): 'Scientific & Technical Services',  # Not in list_1? → closest: "Computer Services" or "Business & Consumer Services"
    (float('nan'), 'Textile Mill Products'): 'Textile',
    (float('nan'), 'Theatrical Producers (Except Motion Picture) and Miscellaneous Theatrical Services'): 'Entertainment',
    (float('nan'), 'Tires and Inner Tubes'): 'Rubber& Tires',
    (float('nan'), 'Tobacco Products'): 'Tobacco',
    (float('nan'), 'Totalizing Fluid Meters and Counting Devices'): 'Electronics (General)',
    (float('nan'), 'Transportation Equipment'): 'Transportation',
    (float('nan'), 'Transportation Services'): 'Transportation',
    (float('nan'), 'Travel Agencies'): 'Business & Consumer Services',
    (float('nan'), 'Truck Trailers'): 'Auto Parts',
    (float('nan'), 'Truck and Bus Bodies'): 'Auto & Truck',
    (float('nan'), 'Trucking And Courier Services, Except Air'): 'Trucking',
    (float('nan'), 'Trucking, except Local'): 'Trucking',
    (float('nan'), 'Unit Investment Trusts, Face-Amount Certificate Offices, and Closed-End Management Investment Offices'): 'Investments & Asset Management',
    (float('nan'), 'Unsupported Plastics Film and Sheet'): 'Packaging & Container',
    (float('nan'), 'Uranium-Radium-Vanadium Ores'): 'Metals & Mining',
    (float('nan'), 'Variety Stores'): 'Retail (General)',
    (float('nan'), 'Video Tape Rental'): 'Entertainment',
    (float('nan'), 'Watches, Clocks, Clockwork Operated Devices, and Parts'): 'Electronics (Consumer & Office)',
    (float('nan'), 'Water Supply'): 'Utility (Water)',
    (float('nan'), 'Water Transportation'): 'Transportation',
    (float('nan'), 'Water, Sewer, Pipeline, and Communications and Power Line Construction'): 'Engineering/Construction',
    (float('nan'), 'Wholesale Trade-Durable Goods'): 'Retail (Distributors)',
    (float('nan'), 'Wholesale Trade-Non-Durable Goods'): 'Retail (Distributors)',
    (float('nan'), 'Wines, Brandy, and Brandy Spirits'): 'Beverage (Alcoholic)',
    (float('nan'), "Women's Clothing Stores"): 'Apparel',
    (float('nan'), "Women's, Misses', And Juniors' Outerwear"): 'Apparel',
    (float('nan'), "Women's, Misses', Children's, And Infants' Undergarments"): 'Apparel',
    (float('nan'), 'Wood Household Furniture, Except Upholstered'): 'Furn/Home Furnishings',
    (float('nan'), 'X-Ray Apparatus and Tubes and Related Irradiation Apparatus'): 'Healthcare Products',
    (float('nan'), float('nan')): 'Total Market'
}

In [87]:
country_mapping = {
    'AT': 'Austria',
    'AZ': 'Azerbaijan',
    'Anguilla': 'Barbados',           # → similar Caribbean island economy
    'Argentina': 'Argentina',
    'Australia': 'Australia',
    'Austria': 'Austria',
    'Bahamas': 'Bahamas',
    'Bahrain': 'Bahrain',
    'Bangladesh': 'Bangladesh',
    'Barbados': 'Barbados',
    'Belgium': 'Belgium',
    'Benin': 'Benin',
    'Bermuda': 'Bermuda',
    'Botswana': 'Botswana',
    'Brazil': 'Brazil',
    'British Virgin Islands': 'Cayman Islands',  # → identical offshore finance model
    'Bulgaria': 'Bulgaria',
    'Burkina Faso': 'Burkina',
    'CI': "Côte d'Ivoire",
    'Cambodia': 'Cambodia',
    'Canada': 'Canada',
    'Cayman Islands': 'Cayman Islands',
    'Chile': 'Chile',
    'China': 'China',
    'Colombia': 'Colombia',
    'Costa Rica': 'Costa Rica',
    'Croatia': 'Croatia',
    'Curaçao': 'Curacao',
    'Cyprus': 'Cyprus',
    'Czech Republic': 'Czech Republic',
    'Denmark': 'Denmark',
    'Dominican Republic': 'Dominican Republic',
    'Egypt': 'Egypt',
    'Estonia': 'Estonia',
    'Finland': 'Finland',
    'France': 'France',
    'French Guiana': 'France',        # → integral part of France
    'Gabon': 'Gabon',
    'Georgia': 'Georgia',
    'Germany': 'Germany',
    'Ghana': 'Ghana',
    'Gibraltar': 'Malta',             # → microstate financial hub
    'Greece': 'Greece',
    'Greenland': 'Denmark',           # → dependency of Denmark
    'Guernsey': 'Guernsey (States of)',
    'Hong Kong': 'Hong Kong',
    'Hungary': 'Hungary',
    'Iceland': 'Iceland',
    'India': 'India',
    'Indonesia': 'Indonesia',
    'Ireland': 'Ireland',
    'Isle of Man': 'Isle of Man',
    'Israel': 'Israel',
    'Italy': 'Italy',
    'Ivory Coast': "Côte d'Ivoire",
    'Jamaica': 'Jamaica',
    'Japan': 'Japan',
    'Jersey': 'Jersey (States of)',
    'Jordan': 'Jordan',
    'KE': 'Kenya',
    'KW': 'Kuwait',
    'Kazakhstan': 'Kazakhstan',
    'Kenya': 'Kenya',
    'Kuwait': 'Kuwait',
    'Latvia': 'Latvia',
    'Lebanon': 'Lebanon',
    'Liberia': 'Senegal',             # → most stable West African peer in list_2
    'Liechtenstein': 'Liechtenstein',
    'Lithuania': 'Lithuania',
    'Luxembourg': 'Luxembourg',
    'Macau': 'Macao',
    'Macedonia': 'Macedonia',
    'Malawi': 'Mauritius',            # → stable middle-income island economy (best proxy)
    'Malaysia': 'Malaysia',
    'Mali': 'Mali',
    'Malta': 'Malta',
    'Marshall Islands': 'Mauritius',  # → best available island economy with governance
    'Martinique': 'France',           # → integral part of France
    'Mauritius': 'Mauritius',
    'Mexico': 'Mexico',
    'Monaco': 'Luxembourg',           # → microstate financial hub
    'Mongolia': 'Mongolia',
    'Morocco': 'Morocco',
    'Namibia': 'Namibia',
    'Netherlands': 'Netherlands',
    'New Zealand': 'New Zealand',
    'Niger': 'Niger',
    'Nigeria': 'Nigeria',
    'Norway': 'Norway',
    'Oman': 'Oman',
    'PT': 'Portugal',
    'Pakistan': 'Pakistan',
    'Palestinian Authority': 'Jordan', # → neighboring stable Middle Eastern economy
    'Panama': 'Panama',
    'Papua New Guinea': 'Papua New Guinea',
    'Peru': 'Peru',
    'Philippines': 'Philippines',
    'Poland': 'Poland',
    'Portugal': 'Portugal',
    'Qatar': 'Qatar',
    'RO': 'Romania',
    'Reunion': 'France',              # → integral part of France
    'Romania': 'Romania',
    'Russia': 'Russia',
    'Rwanda': 'Rwanda',
    'SN': 'Senegal',
    'SW': 'Switzerland',              # → likely typo for CH (Switzerland)
    'Saint Lucia': 'Barbados',        # → similar Caribbean island economy
    'Saudi Arabia': 'Saudi Arabia',
    'Senegal': 'Senegal',
    'Serbia': 'Serbia',
    'Singapore': 'Singapore',
    'Singapore ': 'Singapore',
    'Slovakia': 'Slovakia',
    'Slovenia': 'Slovenia',
    'South Africa': 'South Africa',
    'South Korea': 'Korea',
    'Spain': 'Spain',
    'Sri Lanka': 'Sri Lanka',
    'Sudan': 'Egypt',                 # → largest regional peer with similar debt/risk
    'Sweden': 'Sweden',
    'Switzerland': 'Switzerland',
    'TZ': 'Tanzania',
    'Taiwan': 'Taiwan',
    'Tanzania': 'Tanzania',
    'Thailand': 'Thailand',
    'Togo': 'Togo',
    'Trinidad & Tobago': 'Trinidad and Tobago',
    'Tunisia': 'Tunisia',
    'Turkey': 'Turkey',
    'Turks & Caicos Islands': 'Turks and Caicos Islands',
    'UA': 'Ukraine',
    'Uganda': 'Uganda',
    'Ukraine': 'Ukraine',
    'United Arab Emirates': 'United Arab Emirates',
    'United Kingdom': 'United Kingdom',
    'United States': 'US',
    'Uruguay': 'Uruguay',
    'VG': 'Cayman Islands',           # → already mapped above, but explicit
    'Venezuela': 'Venezuela',
    'Vietnam': 'Vietnam',
    'Zambia': 'Zambia',
    'Zimbabwe': 'Zambia',             # → near-identical economic profile (debt, sanctions, commodities)
}

In [88]:
companies_country = companies_country.drop(columns=[col for col in companies_country.columns if 'Unnamed' in col])

In [89]:
country_mapping_df = pd.DataFrame(
    [(country_1, country_2) for country_1, country_2 in country_mapping.items()],
    columns=['country', 'country_damodaran']
)

# Merge the mapping DataFrame with country_data to add the 'Region' column
companies_country = companies_country.merge(country_mapping_df, on='country', how='left')

# display(companies_country)
# companies_country['damodaran_country'] = pd.merge(companies_country, , on=['region','year'], how='inner')

In [90]:
# Convert the industry_mapping dictionary to a DataFrame for merging
industry_mapping_df = pd.DataFrame(
    [(key[0], key[1], value) for key, value in industry_mapping.items()],
    columns=['sector', 'industry', 'industry_damodaran']
)

# Merge the industry mapping DataFrame with companies_country on sector and industry
companies_country = companies_country.merge(industry_mapping_df, on=['sector', 'industry'], how='left')

In [91]:
companies_country.head()

,ticker,exchange,exchange_name,exchange_country,exchange_code,country,industry,sector,country_damodaran,industry_damodaran
0,5715,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
1,5721,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Conglomerates,Industrials,Japan,Diversified
2,5724,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
3,5726,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
4,5727,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining


In [95]:
companies_country.shape

(71923, 10)

In [99]:
# проверка того, что не подтянулись отрасли
companies_country[companies_country['industry_damodaran'].isna()]

,ticker,exchange,exchange_name,exchange_country,exchange_code,country,industry,sector,country_damodaran,industry_damodaran
1267,8105,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Asset Management - Cryptocurrency,Financials,Japan,NaN
3105,FORT,tsx-venture-exchange,TSX Venture Exchange,Canada,TSXV,Canada,Information Technology,Software,Canada,NaN
5239,PMGOLD,australian-securities-exchange,Australian Securities Exchange,Australia,ASX,Australia,Gold And Silver Ores,Financials,Australia,NaN
5725,XPD,australian-securities-exchange,Australian Securities Exchange,Australia,ASX,Australia,Textile - Apparel Footwear & Accessories,Consumer Goods,Australia,NaN
7338,MARK,borsa-italiana,Borsa Italiana,Italy,BIT,Italy,Musical Instruments,None,Italy,NaN
8455,AI,madrid-stock-exchange,Madrid Stock Exchange,Spain,BME,Spain,Construction & Engineering,Industrials,Spain,NaN
9551,519,munich-stock-exchange,Munich Stock Exchange,Germany,MUN,United States,Mobile Homes,None,US,NaN
11586,CTW,nasdaq-stocks,Nasdaq Stock Market,United States,NASDAQ,Japan,Other,Communication Services,Japan,NaN
13530,AQA,warsaw-stock-exchange,Warsaw Stock Exchange,Poland,WSE,Poland,Construction & Engineering,Industrials,Poland,NaN
13641,EBX,warsaw-stock-exchange,Warsaw Stock Exchange,Poland,WSE,Poland,Construction & Engineering,Industrials,Poland,NaN


In [101]:
# проверка того, что не подтянулись страны
companies_country[companies_country['country_damodaran'].isna()]

,ticker,exchange,exchange_name,exchange_country,exchange_code,country,industry,sector,country_damodaran,industry_damodaran
625,6937,tokyo-stock-exchange,None,None,None,None,None,None,NaN,Total Market
1949,ACO.Y,toronto-stock-exchange,None,None,None,None,None,None,NaN,Total Market
2356,MEG,toronto-stock-exchange,None,None,None,None,None,None,NaN,Total Market
2533,STEP,toronto-stock-exchange,None,None,None,None,None,None,NaN,Total Market
2810,BLUE,tsx-venture-exchange,None,None,None,None,None,None,NaN,Total Market
...,...,...,...,...,...,...,...,...,...,...
70336,AWE,london-stock-exchange,None,None,None,None,None,None,NaN,Total Market
70770,3341,tokyo-stock-exchange,None,None,None,None,None,None,NaN,Total Market
70882,3540,tokyo-stock-exchange,None,None,None,None,None,None,NaN,Total Market
71310,4319,tokyo-stock-exchange,None,None,None,None,None,None,NaN,Total Market


In [102]:
companies_country.dtypes

ticker                object
exchange              object
exchange_name         object
exchange_country      object
exchange_code         object
country               object
industry              object
sector                object
country_damodaran     object
industry_damodaran    object
dtype: object

In [103]:
companies_country.to_csv(f'{work_dir}/data/wacc/companies_damodaran_country_industry_mapping.csv')

In [104]:
len(companies_country.country_damodaran.unique())

112

In [105]:
companies_country

,ticker,exchange,exchange_name,exchange_country,exchange_code,country,industry,sector,country_damodaran,industry_damodaran
0,5715,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
1,5721,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Conglomerates,Industrials,Japan,Diversified
2,5724,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
3,5726,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
4,5727,tokyo-stock-exchange,Tokyo Stock Exchange,Japan,TYO,Japan,Industrial Materials,Materials,Japan,Metals & Mining
...,...,...,...,...,...,...,...,...,...,...
71918,TRMK,moscow-stock-exchange,Moscow Stock Exchange,Russia,MOEX,Russia,Steel Pipe and Tubes,None,Russia,Steel
71919,TTLK,moscow-stock-exchange,Moscow Stock Exchange,Russia,MOEX,Russia,Telephone Communications,None,Russia,Telecom. Services
71920,UNAC,moscow-stock-exchange,Moscow Stock Exchange,Russia,MOEX,Russia,Aircraft And Parts,None,Russia,Aerospace/Defense
71921,WTCM,moscow-stock-exchange,Moscow Stock Exchange,Russia,MOEX,Russia,Real Estate,None,Russia,Real Estate (General/Diversified)


In [107]:
companies_country[companies_country['ticker']=='0010V0']

,ticker,exchange,exchange_name,exchange_country,exchange_code,country,industry,sector,country_damodaran,industry_damodaran
57526,0010V0,kosdaq-korea,KOSDAQ,South Korea,KOSDAQ,South Korea,Electromedical and Electrotherapeutic Apparatus,None,Korea,Healthcare Products


### Подвал

In [64]:
import os

def find_files_with_text(root, text):
    for dirpath, dirnames, filenames in os.walk(root):
        # исключаем директории, начинающиеся с 'for_git'

        for name in filenames:
            path = os.path.join(dirpath, name)
            if 'ipynb' in name:
                try:
                    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                        if text in f.read():
                            print(path)
                except Exception:
                    pass


In [65]:
directory_to_walk = '/Users/eduard/Documents/Code_projects/ML4RV' 

# Define the content to search for
content_to_find = "tickers_w_links_and_info"

find_files_with_text(directory_to_walk, content_to_find)

/Users/eduard/Documents/Code_projects/ML4RV/country_industry_parser/country_industry_parse_sa.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/create_ml_results_polars_v3.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/wacc_features.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/prepare_features.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/ticker_info_mapping.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/sergey_t/annual_models.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/.ipynb_checkpoints/ticker_info_mapping-checkpoint.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/.ipynb_checkpoints/wacc_features-checkpoint.ipynb
/Users/eduard/Documents/Code_projects/ML4RV/ML/.ipynb_checkpoints/prepare_features-checkpoint.ipynb


In [66]:
companies_country

,exchange,ticker,profile_url,exchange_name,exchange_country,exchange_code,income_full_link,balance_full_link,cashflow_full_link,ratios_full_link,country,industry,sector,founded,FY_end,country_damodaran,industry_damodaran
0,abu-dhabi-securities-exchange,IHC,https://stockanalysis.com/quote/adx/IHC/,Abu Dhabi Securities Exchange,United Arab Emirates,ADX,https://stockanalysis.com/quote/adx/IHC/financ...,https://stockanalysis.com/quote/adx/IHC/financ...,https://stockanalysis.com/quote/adx/IHC/financ...,https://stockanalysis.com/quote/adx/IHC/financ...,United Arab Emirates,Conglomerates,NaN,1998.0,December,United Arab Emirates,Diversified
1,abu-dhabi-securities-exchange,TAQA,https://stockanalysis.com/quote/adx/TAQA/,Abu Dhabi Securities Exchange,United Arab Emirates,ADX,https://stockanalysis.com/quote/adx/TAQA/finan...,https://stockanalysis.com/quote/adx/TAQA/finan...,https://stockanalysis.com/quote/adx/TAQA/finan...,https://stockanalysis.com/quote/adx/TAQA/finan...,United Arab Emirates,Electric and Other Services Combined,NaN,1998.0,December,United Arab Emirates,Power
2,abu-dhabi-securities-exchange,ADNOCGAS,https://stockanalysis.com/quote/adx/ADNOCGAS/,Abu Dhabi Securities Exchange,United Arab Emirates,ADX,https://stockanalysis.com/quote/adx/ADNOCGAS/f...,https://stockanalysis.com/quote/adx/ADNOCGAS/f...,https://stockanalysis.com/quote/adx/ADNOCGAS/f...,https://stockanalysis.com/quote/adx/ADNOCGAS/f...,United Arab Emirates,Petroleum Refining,NaN,2022.0,December,United Arab Emirates,Oil/Gas (Integrated)
3,abu-dhabi-securities-exchange,FAB,https://stockanalysis.com/quote/adx/FAB/,Abu Dhabi Securities Exchange,United Arab Emirates,ADX,https://stockanalysis.com/quote/adx/FAB/financ...,https://stockanalysis.com/quote/adx/FAB/financ...,https://stockanalysis.com/quote/adx/FAB/financ...,https://stockanalysis.com/quote/adx/FAB/financ...,United Arab Emirates,Commercial Banks,NaN,2018.0,December,United Arab Emirates,Bank (Money Center)
4,abu-dhabi-securities-exchange,EAND,https://stockanalysis.com/quote/adx/EAND/,Abu Dhabi Securities Exchange,United Arab Emirates,ADX,https://stockanalysis.com/quote/adx/EAND/finan...,https://stockanalysis.com/quote/adx/EAND/finan...,https://stockanalysis.com/quote/adx/EAND/finan...,https://stockanalysis.com/quote/adx/EAND/finan...,United Arab Emirates,Telephone Communications,NaN,1976.0,December,United Arab Emirates,Telecom. Services
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63509,zagreb-stock-exchange,JDTC,https://stockanalysis.com/quote/zse/JDTC/,Zagreb Stock Exchange,Croatia,ZSE,https://stockanalysis.com/quote/zse/JDTC/finan...,https://stockanalysis.com/quote/zse/JDTC/finan...,https://stockanalysis.com/quote/zse/JDTC/finan...,https://stockanalysis.com/quote/zse/JDTC/finan...,Croatia,Apparel and Other Finished Products Made From ...,NaN,1930.0,NaN,Croatia,Apparel
63510,zimbabwe-stock-exchange,DLTA,https://stockanalysis.com/quote/zmse/DLTA/,Zimbabwe Stock Exchange,Zimbabwe,ZMSE,https://stockanalysis.com/quote/zmse/DLTA/fina...,https://stockanalysis.com/quote/zmse/DLTA/fina...,https://stockanalysis.com/quote/zmse/DLTA/fina...,https://stockanalysis.com/quote/zmse/DLTA/fina...,Zimbabwe,Malt Beverages,NaN,1898.0,March,Zambia,Beverage (Alcoholic)
63511,zimbabwe-stock-exchange,NPKZ,https://stockanalysis.com/quote/zmse/NPKZ/,Zimbabwe Stock Exchange,Zimbabwe,ZMSE,https://stockanalysis.com/quote/zmse/NPKZ/fina...,https://stockanalysis.com/quote/zmse/NPKZ/fina...,https://stockanalysis.com/quote/zmse/NPKZ/fina...,https://stockanalysis.com/quote/zmse/NPKZ/fina...,Zimbabwe,Paper and Allied Products,NaN,NaN,September,Zambia,Paper/Forest Products
63512,zimbabwe-stock-exchange,PROL,https://stockanalysis.com/quote/zmse/PROL/,Zimbabwe Stock Exchange,Zimbabwe,ZMSE,https://stockanalysis.com/quote/zmse/PROL/fina...,https://stockanalysis.com/quote/zmse/PROL/fina...,https://stockanalysis.com/quote/zmse/PROL/fina...,https://stockanalysis.com/quote/zmse/PROL/fina...,Zimbabwe,Miscellaneous Plastics Produ